# URA Tax Assistant - Data Ingestion & Augmentation Pipeline

## Overview

This notebook implements the complete data pipeline for the URA (Uganda Revenue Authority) Tax Assistant chatbot. It processes data from multiple sources, generates synthetic QA pairs using a teacher LLM, augments data into multiple training formats, and produces comprehensive EDA visualizations.

### Pipeline Stages

```
1. Install & Import    ->  Dependencies, GPU detection
2. Configuration       ->  Paths, hyperparameters, thresholds
3. Data Loading        ->  CSVs, PDFs, Teacher QA, Luganda translations
4. Data Validation     ->  Schema checks with Pandera
5. Teacher QA Gen      ->  LLM-generated QA from PDF chunks (4-bit quantized)
6. Augmentation        ->  Multi-format output (instruction, chat, Gemma)
7. Export              ->  JSONL, train/val splits, statistics
8. EDA                 ->  Visualizations, quality assessment, reports
```

### Environment
- **Runtime:** Kaggle Notebooks (2x T4 GPU)
- **Model:** Qwen2.5-7B-Instruct (4-bit NF4 quantization)
- **Data:** URA FAQs, Tax PDFs, Luganda-English parallel corpus

In [ ]:
# Install Dependencies (pinned for reproducibility - 2026 production standards)
!pip install -q --no-warn-script-location pymupdf==1.25.3 pymupdf4llm==0.0.17 datasets==3.3.2 sentencepiece==0.2.0 bitsandbytes==0.45.1 accelerate==1.3.0 pandera==0.21.1 transformers==4.48.3 datasketch==1.6.5 pyarrow==18.1.0
!pip install -q --no-warn-script-location torch==2.5.1 --index-url https://download.pytorch.org/whl/cu118

# Environment variables for GPU optimization
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:128"
os.environ["HF_HUB_OFFLINE"] = "0"
os.environ["TRANSFORMERS_CACHE"] = "/kaggle/working/cache"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTHONHASHSEED"] = "42"

In [ ]:
# Import Libraries and Setup (with global reproducibility)
import json
import gc
import os
import random
import re
import sys
import logging
import time
from pathlib import Path
from typing import Optional, List, Dict, Any, Tuple
from datetime import datetime, timezone
from dataclasses import dataclass, field
from enum import Enum
from contextlib import contextmanager
import hashlib

import pandas as pd
import numpy as np
import torch
from tqdm import tqdm
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# ---------- Global Reproducibility ----------
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
os.environ["PYTHONHASHSEED"] = str(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Import pymupdf for PDF processing
try:
    import pymupdf as fitz
    import pymupdf4llm
    PYPDF_AVAILABLE = True
except ImportError:
    PYPDF_AVAILABLE = False
    print("Warning: pymupdf4llm not installed. PDF processing will be limited.")

# Import datasketch for semantic dedup
try:
    from datasketch import MinHash, MinHashLSH
    MINHASH_AVAILABLE = True
except ImportError:
    MINHASH_AVAILABLE = False
    print("Warning: datasketch not installed. Semantic dedup will use exact-hash only.")

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA devices: {torch.cuda.device_count()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
        print(f"    Memory: {torch.cuda.get_device_properties(i).total_mem / 1e9:.1f}GB")

## Configuration

Defines all pipeline parameters in a single `KaggleConfig` dataclass:
- **Data paths**: Auto-loaded from Kaggle dataset via `kagglehub`
- **Processing**: Chunk size, overlap, text length limits
- **Teacher model**: Model name, generation params, batch sizing
- **Quality thresholds**: Confidence minimums, similarity ceilings, deduplication
- **Output**: Directory paths, train/val split ratio

In [ ]:
# Cell 3: Configuration Classes
class DataType(str, Enum):
    """Enum for data types."""
    FAQ = "faq"
    PDF = "pdf"
    TRANSLATION = "translation"
    TEACHER_QA = "teacher_qa"
    SYNTHETIC = "synthetic"

class FormatType(str, Enum):
    """Enum for format types."""
    INSTRUCTION = "instruction"
    CHAT = "chat"
    GEMMA = "gemma"
    HF_DATASET = "hf_dataset"

@dataclass
class GeneratedQA:
    """Enhanced QA generation with confidence scoring."""
    question: str
    answer: str
    chunk_text: str
    source_pdf: str
    chunk_id: int
    question_type: str
    confidence: float = 1.0
    
    def to_dict(self):
        return {
            "question": self.question,
            "answer": self.answer,
            "chunk_text": self.chunk_text[:500],  # Truncate for storage
            "source_pdf": self.source_pdf,
            "chunk_id": self.chunk_id,
            "question_type": self.question_type,
            "confidence": self.confidence
        }

@dataclass
class KaggleConfig:
    """Kaggle-specific configuration."""
    # Kaggle paths (adjust based on your dataset structure)
    kaggle_dataset_name = "mpairwe/ura-dataset"
    
    # Base paths after loading dataset
    @property
    def base_path(self):
        import kagglehub
        return Path(kagglehub.load_dataset(self.kaggle_dataset_name))
    
    @property
    def data_root(self):
        return self.base_path / "Data"
    
    @property
    def paths(self):
        return {
            "ura_pdfs": self.data_root / "pdfs",
            "ura_faqs": self.data_root / "dataset",
            "en_lg_ttt": self.data_root / "TTT",
            "lg_audio": self.data_root / "lgaudio",
            "teacher_qa": self.data_root / "teacher_qa",
        }
    
    # Processing
    random_seed: int = 42
    chunk_size: int = 500
    chunk_overlap: int = 50
    max_text_length: int = 2000
    min_text_length: int = 50
    
    # Teacher Model (Optimized for 2x T4 GPUs on Kaggle)
    teacher_model: str = "Qwen/Qwen2.5-7B-Instruct"
    max_new_tokens: int = 200
    temperature: float = 0.7
    top_p: float = 0.95
    batch_size_per_gpu: int = 2  # Reduced for 7B model on T4
    
    # Augmentation
    questions_per_chunk: int = 3
    augment_factor: int = 2
    max_variations: int = 5
    
    # Quality thresholds
    min_question_words: int = 3
    min_answer_words: int = 10
    min_chunk_words: int = 20
    max_duplicate_similarity: float = 0.9
    min_qa_confidence: float = 0.6
    fallback_qa_confidence: float = 0.7
    qa_relevance_threshold: float = 0.3
    qa_similarity_ceiling: float = 0.8
    default_qa_confidence: float = 0.8
    
    # Output
    output_dir: Path = Path("/kaggle/working/artifacts")
    train_val_split: float = 0.1
    export_all_formats: bool = True
    compress_output: bool = False
    
    # Performance
    use_multiprocessing: bool = True
    cache_intermediate: bool = True
    
    # Language
    support_luganda: bool = True
    create_bilingual_pairs: bool = True
    

    # Provenance & Security
    dataset_version: str = "2026.02"  # Semantic version for the dataset
    require_hash_verification: bool = True
    trusted_sources: tuple = ("mpairwe/ura-dataset", "mpairwelauben/ura-training-data", "mpairwelauben/ura-tax-data-v2")
    
    # PII Redaction
    redact_pii: bool = True
    pii_patterns: dict = field(default_factory=lambda: {
        'email': r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}',
        'phone_ug': r'(?:\+256|0)[0-9]{9}',
        'phone_intl': r'\+?[0-9]{10,15}',
        'national_id': r'[A-Z]{2}[0-9]{7,14}',
        'tin_ug': r'[0-9]{10,12}',
    })
    
    # Semantic dedup (MinHash)
    minhash_threshold: float = 0.7  # Jaccard similarity threshold for near-duplicates
    minhash_num_perm: int = 128
    
    # Generation determinism
    generation_seed: int = 42  # Separate seed for teacher model generation

    def __post_init__(self):
        """Create directories and set random seeds."""
        random.seed(self.random_seed)
        np.random.seed(self.random_seed)
        torch.manual_seed(self.random_seed)
        os.environ["PYTHONHASHSEED"] = str(self.random_seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(self.random_seed)
        
        # Create directories
        self.output_dir.mkdir(parents=True, exist_ok=True)
        
        # Set cache directory
        self.cache_dir = Path("/kaggle/working/models")
        self.cache_dir.mkdir(parents=True, exist_ok=True)

# Initialize config
config = KaggleConfig()
print("📂 Configuration initialized")
print(f"📁 Output directory: {config.output_dir}")
print(f"📁 Cache directory: {config.cache_dir}")

In [ ]:
# Cell 4: Setup Logging
def setup_logging(verbose: bool = False, log_file: Optional[Path] = None) -> logging.Logger:
    """Setup standardized logging."""
    logger = logging.getLogger(__name__)
    logger.setLevel(logging.DEBUG if verbose else logging.INFO)
    
    # Clear existing handlers
    logger.handlers.clear()
    
    # Console handler
    console_handler = logging.StreamHandler(sys.stdout)
    console_handler.setLevel(logging.DEBUG if verbose else logging.INFO)
    console_format = logging.Formatter(
        '%(asctime)s - %(name)s - %(levelname)s - %(message)s',
        datefmt='%H:%M:%S'
    )
    console_handler.setFormatter(console_format)
    logger.addHandler(console_handler)
    
    # File handler if specified
    if log_file:
        file_handler = logging.FileHandler(log_file)
        file_handler.setLevel(logging.DEBUG)
        file_format = logging.Formatter(
            '%(asctime)s - %(name)s - %(levelname)s - %(filename)s:%(lineno)d - %(message)s'
        )
        file_handler.setFormatter(file_format)
        logger.addHandler(file_handler)
    
    return logger

logger = setup_logging(verbose=True, log_file=config.output_dir / "data_pipeline.log")
logger.info(" Starting URA Tax Assistant Data Pipeline")
logger.info(f"Timestamp: {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S')}")

## Text Processing

`TextProcessor` provides stateless utility methods for text cleaning, similarity calculation, QA validation, hashing, and key term extraction. All thresholds reference `KaggleConfig`.

In [ ]:
# Cell 5: Text Processor
class TextProcessor:
    """Standardized text processing utilities."""
    
    @staticmethod
    def clean_text(text: str, preserve_paragraphs: bool = True) -> str:
        """Clean and normalize text with options for formatting preservation."""
        if pd.isna(text) or not text:
            return ""
        
        text = str(text)
        
        # Remove control characters and BOM
        text = re.sub(r'[\x00-\x08\x0B\x0C\x0E-\x1F\x7F]', '', text)
        text = text.replace('\ufeff', '').replace('\u200b', '')
        
        if preserve_paragraphs:
            # Preserve meaningful paragraph breaks
            text = re.sub(r'\n\s*\n+', '\n\n', text)
            text = re.sub(r'[ \t]+', ' ', text)
        else:
            # Standard cleaning
            text = re.sub(r'\s+', ' ', text)
        
        # Normalize quotes and dashes
        text = text.replace('``', '"').replace("''", '"')
        text = re.sub(r'[\u2018\u2019]', "'", text)
        text = re.sub(r'[\u201C\u201D]', '"', text)
        text = re.sub(r'[\u2013\u2014]', '-', text)
        
        # Remove excessive punctuation
        text = re.sub(r'([.!?])\1+', r'\1', text)
        
        # Remove special characters but keep common punctuation
        text = re.sub(r'[^\w\s.,;:!?\'\"()\-–—/#$%&@\d]', '', text)
        
        # Remove leading/trailing whitespace
        text = text.strip()
        
        return text
    
    @staticmethod
    def calculate_similarity(text1: str, text2: str) -> float:
        """Calculate Jaccard similarity between two texts."""
        if not text1 or not text2:
            return 0.0
        
        # Create sets of words (case-insensitive)
        set1 = set(text1.lower().split())
        set2 = set(text2.lower().split())
        
        if not set1 or not set2:
            return 0.0
        
        intersection = len(set1.intersection(set2))
        union = len(set1.union(set2))
        
        return intersection / union if union > 0 else 0.0
    
    @staticmethod
    def validate_qa_pair(question: str, answer: str, 
                        min_q_words: int = 3, 
                        min_a_words: int = 5) -> Tuple[bool, str]:
        """Validate quality of QA pair."""
        if not question or not answer:
            return False, "Empty question or answer"
        
        q_words = len(question.split())
        a_words = len(answer.split())
        
        if q_words < min_q_words:
            return False, f"Question too short ({q_words} < {min_q_words} words)"
        
        if a_words < min_a_words:
            return False, f"Answer too short ({a_words} < {min_a_words} words)"
        
        # Check for common issues
        if question.strip().lower() == answer.strip().lower()[:len(question)]:
            return False, "Answer repeats question"
        
        if TextProcessor.calculate_similarity(question, answer) > config.qa_similarity_ceiling:
            return False, "Question and answer too similar"
        
        return True, "Valid"
    
    @staticmethod
    def generate_hash(text: str) -> str:
        """Generate SHA256 hash for text deduplication."""
        return hashlib.sha256(text.encode('utf-8')).hexdigest()[:16]
    
    @staticmethod
    def extract_topic_from_text(text: str, max_words: int = 5) -> str:
        """Extract a topic phrase from text for question generation."""
        # Get first meaningful sentence
        sentences = re.split(r'[.!?]', text)
        if sentences:
            for sentence in sentences:
                sentence = sentence.strip()
                if len(sentence.split()) >= 3:
                    words = sentence.split()[:max_words]
                    topic = ' '.join(words).lower()
                    topic = re.sub(r'[^\w\s]', '', topic)
                    return topic
        
        # Fallback: first N words
        words = text.split()[:max_words]
        topic = ' '.join(words).lower()
        topic = re.sub(r'[^\w\s]', '', topic)
        return topic
    
    @staticmethod

    @staticmethod
    def redact_pii(text: str, patterns: dict = None) -> str:
        """Redact personally identifiable information from text.
        
        Replaces emails, phone numbers, national IDs, and TINs with placeholders.
        """
        if not text or not isinstance(text, str):
            return text
        
        default_patterns = {
            'email': r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}',
            'phone_ug': r'(?:\+256|0)[0-9]{9}',
            'phone_intl': r'\+?[0-9]{10,15}',
            'national_id': r'[A-Z]{2}[0-9]{7,14}',
            'tin_ug': r'[0-9]{10,12}',
        }
        
        pii_patterns = patterns or default_patterns
        redacted = text
        for pii_type, pattern in pii_patterns.items():
            redacted = re.sub(pattern, f'[REDACTED_{pii_type.upper()}]', redacted)
        return redacted

    def extract_key_terms(text: str) -> List[str]:
        """Extract key tax-related terms from text."""
        # Tax-specific terminology
        tax_terms = {
            'vat', 'tin', 'withholding', 'corporation', 'individual', 'business',
            'compliance', 'payment', 'return', 'assessment', 'customs', 'duty',
            'excise', 'income', 'property', 'stamp', 'taxpayer', 'registration',
            'filing', 'deadline', 'penalty', 'exemption', 'deduction', 'credit',
            'tax', 'ura', 'authority', 'revenue', 'uganda'
        }
        
        # Find terms in text
        found_terms = []
        text_lower = text.lower()
        
        for term in tax_terms:
            if term in text_lower:
                found_terms.append(term)
        
        # Also extract capitalized phrases (potential proper nouns)
        capitalized = re.findall(r'\b[A-Z][a-z]+(?:\s+[A-Z][a-z]+)*\b', text)
        found_terms.extend(capitalized[:3])
        
        return list(set(found_terms))[:8]  # Limit to 8 terms

processor = TextProcessor()

## Data Loading

`EnhancedKaggleDataLoader` loads from 5 sources:

| Source | Format | Content |
|--------|--------|---------|
| `Data/dataset/` | CSV/XLSX | URA FAQ pairs (45 files) |
| `Data/pdfs/` | PDF | Tax guides, handbooks (47 files, 150MB) |
| `Data/teacher_qa/` | JSONL | Pre-generated synthetic QA pairs |
| `Data/TTT/` | CSV/TXT | Luganda-English parallel corpus |
| `Data/lgaudio/` | JSON/TXT | Audio transcription data |

In [ ]:
# Cell 6: Enhanced Data Loader with Translation Support
class EnhancedKaggleDataLoader:
    """Enhanced data loader with better translation support."""
    
    def __init__(self, config: KaggleConfig, logger: logging.Logger):
        self.config = config
        self.logger = logger
        self.processor = TextProcessor()
        
        # Verify paths exist
        self.logger.info("🔍 Checking dataset paths...")
        for name, path in config.paths.items():
            if path.exists():
                self.logger.info(f"  ✓ {name}: {path}")
            else:
                self.logger.warning(f"  ✗ {name}: Not found at {path}")
    
    def load_all_data(self):
        """Load all data sources."""
        self.logger.info("📂 Loading all data sources...")
        
        # Load FAQ CSVs
        faq_df = self.load_csv_faqs()
        
        # Load PDF chunks
        pdf_chunks = self.load_pdf_content()
        
        # Load teacher QA if exists
        teacher_qa_df = self.load_teacher_qa()
        
        # Load translations if enabled
        translations_df = pd.DataFrame()
        if self.config.support_luganda:
            translations_df = self.load_luganda_translations()
        
        return faq_df, pdf_chunks, teacher_qa_df, translations_df
    
    def load_csv_faqs(self) -> pd.DataFrame:
        """Load all CSV/XLSX FAQ files."""
        faq_dir = self.config.paths["ura_faqs"]
        if not faq_dir.exists():
            self.logger.warning(f"FAQ directory not found: {faq_dir}")
            return pd.DataFrame()
        
        data_files = sorted(
            list(faq_dir.glob("*.csv")) + 
            list(faq_dir.glob("*.xlsx"))
        )
        
        self.logger.info(f"Found {len(data_files)} FAQ files")
        
        frames = []
        for path in data_files:
            try:
                df = self._read_file_with_fallback(path)
                if df.empty:
                    continue
                
                # Process columns
                processed_df = self._process_faq_columns(df, path)
                if processed_df is not None:
                    frames.append(processed_df)
                    
            except Exception as e:
                self.logger.error(f"Error loading {path.name}: {e}")
        
        if frames:
            result = pd.concat(frames, ignore_index=True)
            self.logger.info(f"✓ Loaded {len(result)} FAQ Q/A pairs")
            return result
        
        return pd.DataFrame(columns=['question', 'answer', 'source', 'category', 'data_type'])
    
    def _read_file_with_fallback(self, path: Path) -> pd.DataFrame:
        """Read file with multiple fallback strategies."""
        encodings = ['utf-8-sig', 'utf-8', 'latin-1', 'cp1252', 'iso-8859-1']
        
        for enc in encodings:
            try:
                if path.suffix == '.csv':
                    df = pd.read_csv(path, encoding=enc, on_bad_lines='skip')
                else:
                    df = pd.read_excel(path, engine='openpyxl')
                
                if not df.empty:
                    return df
            except Exception:
                continue
        
        # Final attempt
        try:
            df = pd.read_csv(path, sep=None, engine='python', on_bad_lines='skip')
            return df
        except Exception:
            return pd.DataFrame()
    
    def _process_faq_columns(self, df: pd.DataFrame, path: Path) -> Optional[pd.DataFrame]:
        """Process and standardize FAQ columns."""
        # Clean column names
        df.columns = [str(col).strip().lower() for col in df.columns]
        
        # Find question and answer columns
        question_candidates = {'question', 'questions', 'q', 'query'}
        answer_candidates = {'answer', 'answers', 'a', 'response'}
        
        q_col = next((c for c in df.columns if c in question_candidates), None)
        a_col = next((c for c in df.columns if c in answer_candidates), None)
        
        if not q_col or not a_col:
            if len(df.columns) >= 2:
                q_col, a_col = df.columns[0], df.columns[1]
                self.logger.warning(f"Using first two columns as Q/A for {path.name}")
            else:
                return None
        
        # Select and rename columns
        df = df[[q_col, a_col]].copy()
        df.columns = ['question', 'answer']
        
        # Clean text
        df['question'] = df['question'].apply(lambda x: self.processor.clean_text(x, False))
        df['answer'] = df['answer'].apply(lambda x: self.processor.clean_text(x, False))
        
        # Filter empty rows
        df = df[
            (df['question'].str.len() > 10) & 
            (df['answer'].str.len() > 10)
        ]
        
        if not df.empty:
            df['source'] = path.name
            df['category'] = path.stem.replace('ura_', '').replace('_faqs', '').replace('_', ' ').replace('-', ' ').lower()
            df['data_type'] = DataType.FAQ.value
            df['hash'] = df.apply(
                lambda row: self.processor.generate_hash(f"{row['question']}{row['answer']}"), 
                axis=1
            )
            return df
        
        return None
    
    def load_pdf_content(self) -> List[Dict[str, Any]]:
        """Load and chunk PDF content."""
        if not PYPDF_AVAILABLE:
            self.logger.warning("Skipping PDF loading (pymupdf not available)")
            return []
        
        pdf_dir = self.config.paths["ura_pdfs"]
        if not pdf_dir.exists():
            self.logger.warning(f"PDF directory not found: {pdf_dir}")
            return []
        
        pdf_files = sorted(pdf_dir.glob("*.pdf"))
        self.logger.info(f"Found {len(pdf_files)} PDF files")
        
        all_chunks = []
        for pdf_file in pdf_files:
            try:
                chunks = self._extract_pdf_chunks(pdf_file)
                all_chunks.extend(chunks)
                self.logger.info(f"  {pdf_file.name}: {len(chunks)} chunks")
            except Exception as e:
                self.logger.error(f"Error processing {pdf_file.name}: {e}")
        
        self.logger.info(f"✓ Total PDF chunks: {len(all_chunks)}")
        return all_chunks
    
    def _extract_pdf_chunks(self, pdf_path: Path) -> List[Dict[str, Any]]:
        """Extract and chunk a single PDF file with page-level metadata."""
        try:
            # Try page-level extraction first (layout-aware)
            try:
                md_pages = pymupdf4llm.to_markdown(
                    str(pdf_path), pages=None, show_progress=False, page_chunks=True
                )
            except TypeError:
                # Fallback: single-chunk extraction for older pymupdf4llm
                md_pages = None
            
            if md_pages and isinstance(md_pages, list):
                # Page-level extraction succeeded
                pdf_chunks = []
                for page_idx, page_data in enumerate(md_pages):
                    page_text = page_data if isinstance(page_data, str) else page_data.get('text', str(page_data))
                    page_text = self.processor.clean_text(page_text, preserve_paragraphs=True)
                    
                    if len(page_text) < 50:
                        continue
                    
                    # Section detection from headings
                    section = self._detect_section(page_text)
                    
                    # Chunk this page
                    chunks = self._smart_chunk_text(page_text)
                    key_terms = self.processor.extract_key_terms(page_text)
                    
                    for i, chunk in enumerate(chunks):
                        words = chunk.split()
                        if len(words) >= self.config.min_chunk_words:
                            pdf_chunks.append({
                                'text': chunk,
                                'source': pdf_path.name,
                                'page': page_idx + 1,
                                'section': section,
                                'chunk_id': len(pdf_chunks),
                                'total_chunks': 0,  # Updated after
                                'category': pdf_path.stem.replace('-', ' ').replace('_', ' ').lower(),
                                'key_terms': key_terms,
                                'data_type': DataType.PDF.value,
                                'word_count': len(words),
                                'hash': self.processor.generate_hash(chunk)
                            })
                
                # Update total_chunks
                for chunk in pdf_chunks:
                    chunk['total_chunks'] = len(pdf_chunks)
                
                return pdf_chunks
            
            # Fallback: single-document extraction
            md_result = pymupdf4llm.to_markdown(str(pdf_path), pages=None, show_progress=False)
            if isinstance(md_result, list):
                text = self.processor.clean_text("\n\n".join(
                    chunk.get('text', str(chunk)) if isinstance(chunk, dict) else str(chunk)
                    for chunk in md_result
                ), preserve_paragraphs=True)
            else:
                text = self.processor.clean_text(str(md_result), preserve_paragraphs=True)
            
            if len(text) < 100:
                self.logger.warning(f"Insufficient text from {pdf_path.name}")
                return []
            
            # Smart chunking
            chunks = self._smart_chunk_text(text)
            
            # Extract key terms for each chunk
            key_terms = self.processor.extract_key_terms(text)
            
            # Create chunk metadata
            pdf_chunks = []
            for i, chunk in enumerate(chunks):
                words = chunk.split()
                if len(words) >= self.config.min_chunk_words:
                    pdf_chunks.append({
                        'text': chunk,
                        'source': pdf_path.name,
                        'chunk_id': i,
                        'total_chunks': len(chunks),
                        'category': pdf_path.stem.replace('-', ' ').replace('_', ' ').lower(),
                        'key_terms': key_terms,
                        'data_type': DataType.PDF.value,
                        'word_count': len(words),
                        'hash': self.processor.generate_hash(chunk)
                    })
            
            return pdf_chunks
            
        except Exception as e:
            self.logger.error(f"Error extracting {pdf_path.name}: {e}")
            return []
    

    def _detect_section(self, text: str) -> str:
        """Detect document section from content patterns."""
        patterns = [
            (r'(?i)\bVAT\b', 'vat'), (r'(?i)\bincome\s*tax\b', 'income_tax'),
            (r'(?i)\bTIN\b', 'tin'), (r'(?i)\bcustoms\b', 'customs'),
            (r'(?i)\bexcise\b', 'excise'), (r'(?i)\bpenalt', 'penalties'),
            (r'(?i)\bregistration\b', 'registration'), (r'(?i)\brefund\b', 'refund'),
            (r'(?i)\bfiling\b', 'filing'), (r'(?i)\bpayment\b', 'payment'),
        ]
        for pat, section in patterns:
            if re.search(pat, text[:500]):
                return section
        return 'general'

    def _smart_chunk_text(self, text: str) -> List[str]:
        """Split text into semantically meaningful chunks."""
        if not text or len(text.strip()) < 50:
            return []
        
        # Preserve paragraph structure
        paragraphs = [p.strip() for p in text.split('\n\n') if p.strip()]
        
        chunks = []
        current_chunk = []
        current_length = 0
        
        for paragraph in paragraphs:
            para_len = len(paragraph)
            
            if current_length + para_len > self.config.chunk_size and current_chunk:
                chunk_text = '\n\n'.join(current_chunk)
                if len(chunk_text) > 30:
                    chunks.append(chunk_text)
                
                # Keep last paragraph for overlap
                if self.config.chunk_overlap > 0 and current_chunk:
                    overlap_paras = []
                    overlap_len = 0
                    for p in reversed(current_chunk):
                        if overlap_len + len(p) <= self.config.chunk_overlap:
                            overlap_paras.insert(0, p)
                            overlap_len += len(p)
                        else:
                            break
                    current_chunk = overlap_paras
                    current_length = overlap_len
                else:
                    current_chunk = []
                    current_length = 0
            
            current_chunk.append(paragraph)
            current_length += para_len
        
        # Add final chunk
        if current_chunk:
            chunk_text = '\n\n'.join(current_chunk)
            if len(chunk_text) > 30:
                chunks.append(chunk_text)
        
        # Filter out very small chunks
        chunks = [c for c in chunks if len(c) > 30]
        
        return chunks
    
    def load_teacher_qa(self) -> pd.DataFrame:
        """Load teacher-generated QA pairs."""
        qa_dir = self.config.paths["teacher_qa"]
        if not qa_dir.exists():
            self.logger.info(f"Teacher QA directory not found: {qa_dir}")
            return pd.DataFrame()
        
        qa_files = sorted(qa_dir.glob("*.jsonl"))
        self.logger.info(f"Found {len(qa_files)} teacher QA files")
        
        all_qa = []
        for qa_file in qa_files:
            try:
                with open(qa_file, 'r', encoding='utf-8') as f:
                    for line in f:
                        try:
                            qa = json.loads(line.strip())
                            standardized = {
                                'question': qa.get('question', ''),
                                'answer': qa.get('answer', ''),
                                'source': qa.get('source', qa_file.name),
                                'category': qa.get('category', 'teacher_generated'),
                                'data_type': DataType.TEACHER_QA.value,
                                'confidence': qa.get('confidence', 1.0),
                                'question_type': qa.get('question_type', 'factual')
                            }
                            
                            is_valid, _ = self.processor.validate_qa_pair(
                                standardized['question'], standardized['answer']
                            )
                            if is_valid:
                                standardized['hash'] = self.processor.generate_hash(
                                    f"{standardized['question']}{standardized['answer']}"
                                )
                                all_qa.append(standardized)
                                
                        except json.JSONDecodeError:
                            continue
                            
            except Exception as e:
                self.logger.error(f"Error reading {qa_file.name}: {e}")
        
        if all_qa:
            df = pd.DataFrame(all_qa)
            self.logger.info(f"✓ Loaded {len(df)} teacher-generated Q/A pairs")
            return df
        
        return pd.DataFrame()
    
    def load_luganda_translations(self) -> pd.DataFrame:
        """Load English-Luganda parallel data from TTT and lgaudio."""
        translations = []
        
        # Load from TTT directory
        ttt_translations = self._load_ttt_translations()
        translations.extend(ttt_translations)
        
        # Load from lgaudio directory
        audio_translations = self._load_lgaudio_translations()
        translations.extend(audio_translations)
        
        result = pd.DataFrame(translations) if translations else pd.DataFrame()
        self.logger.info(f" Total translation pairs loaded: {len(result)}")
        
        # Verify translation quality
        if not result.empty:
            self._verify_translation_quality(result)
        
        return result
    
    def _load_ttt_translations(self) -> List[Dict]:
        """Load translations from TTT directory."""
        ttt_dir = self.config.paths["en_lg_ttt"]
        if not ttt_dir.exists():
            self.logger.warning(f"TTT directory not found: {ttt_dir}")
            return []
        
        ttt_files = sorted(
            list(ttt_dir.glob("*.csv")) + 
            list(ttt_dir.glob("*.xlsx")) + 
            list(ttt_dir.glob("*.txt"))
        )
        
        self.logger.info(f"📁 Found {len(ttt_files)} TTT translation files")
        
        translations = []
        for path in ttt_files:
            try:
                df = self._read_ttt_file(path)
                if df.empty:
                    continue
                
                # Process each row
                for _, row in df.iterrows():
                    if 'english' in row and 'luganda' in row:
                        en_text = self.processor.clean_text(str(row['english']))
                        lg_text = self.processor.clean_text(str(row['luganda']))
                        
                        if self._is_valid_translation(en_text, lg_text):
                            translations.append({
                                'english': en_text,
                                'luganda': lg_text,
                                'source': f"ttt_{path.name}",
                                'data_type': DataType.TRANSLATION.value,
                                'hash': self.processor.generate_hash(f"{en_text}{lg_text}")
                            })
                
                self.logger.info(f"  ✓ {path.name}: loaded translations")
                        
            except Exception as e:
                self.logger.error(f"  ✗ Error loading TTT file {path.name}: {e}")
        
        return translations
    
    def _read_ttt_file(self, path: Path) -> pd.DataFrame:
        """Read TTT file with format detection."""
        if path.suffix == '.csv':
            # Try different encodings and separators
            for sep in [',', ';', '\t']:
                try:
                    df = pd.read_csv(path, sep=sep, encoding='utf-8')
                    if len(df.columns) >= 2:
                        return self._standardize_ttt_columns(df)
                except Exception:
                    continue
        
        elif path.suffix == '.xlsx':
            try:
                df = pd.read_excel(path, engine='openpyxl')
                return self._standardize_ttt_columns(df)
            except Exception:
                pass
        
        elif path.suffix == '.txt':
            try:
                # Try tab-separated
                df = pd.read_csv(path, sep='\t', encoding='utf-8')
                return self._standardize_ttt_columns(df)
            except Exception:
                # Try reading line by line
                with open(path, 'r', encoding='utf-8') as f:
                    lines = f.readlines()
                
                data = []
                for line in lines:
                    if '|' in line:
                        parts = line.strip().split('|')
                    elif '\t' in line:
                        parts = line.strip().split('\t')
                    else:
                        parts = line.strip().split(',')
                    
                    if len(parts) >= 2:
                        data.append({'english': parts[0], 'luganda': parts[1]})
                
                return pd.DataFrame(data)
        
        return pd.DataFrame()
    
    def _standardize_ttt_columns(self, df: pd.DataFrame) -> pd.DataFrame:
        """Standardize column names for TTT files."""
        df.columns = [str(col).strip().lower() for col in df.columns]
        
        # Map common column names
        column_mapping = {}
        for col in df.columns:
            col_lower = col.lower()
            if any(x in col_lower for x in ['english', 'en', 'eng']):
                column_mapping[col] = 'english'
            elif any(x in col_lower for x in ['luganda', 'lg', 'lug', 'luga']):
                column_mapping[col] = 'luganda'
        
        df = df.rename(columns=column_mapping)
        
        # Ensure we have both columns
        if 'english' not in df.columns and len(df.columns) >= 1:
            df = df.rename(columns={df.columns[0]: 'english'})
        if 'luganda' not in df.columns and len(df.columns) >= 2:
            df = df.rename(columns={df.columns[1]: 'luganda'})
        
        return df
    
    def _load_lgaudio_translations(self) -> List[Dict]:
        """Load translations from lgaudio directory."""
        audio_dir = self.config.paths["lg_audio"]
        if not audio_dir.exists():
            self.logger.warning(f"lgaudio directory not found: {audio_dir}")
            return []
        
        audio_files = sorted(
            list(audio_dir.glob("*.csv")) + 
            list(audio_dir.glob("*.txt")) +
            list(audio_dir.glob("*.json"))
        )
        
        self.logger.info(f"📁 Found {len(audio_files)} lgaudio files")
        
        translations = []
        for path in audio_files:
            try:
                if path.suffix == '.json':
                    audio_translations = self._read_audio_json(path)
                else:
                    audio_translations = self._read_audio_text_file(path)
                
                translations.extend(audio_translations)
                self.logger.info(f"  ✓ {path.name}: loaded translations")
                        
            except Exception as e:
                self.logger.error(f"  ✗ Error loading audio file {path.name}: {e}")
        
        return translations
    
    def _read_audio_json(self, path: Path) -> List[Dict]:
        """Read audio translations from JSON file."""
        with open(path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        
        translations = []
        
        # Handle different JSON structures
        if isinstance(data, list):
            for item in data:
                if isinstance(item, dict):
                    en_text = item.get('text_en') or item.get('english') or item.get('en')
                    lg_text = item.get('text_lg') or item.get('luganda') or item.get('lg')
                    
                    if en_text and lg_text:
                        en_clean = self.processor.clean_text(str(en_text))
                        lg_clean = self.processor.clean_text(str(lg_text))
                        
                        if self._is_valid_translation(en_clean, lg_clean):
                            translations.append({
                                'english': en_clean,
                                'luganda': lg_clean,
                                'source': f"audio_{path.name}",
                                'data_type': DataType.TRANSLATION.value,
                                'hash': self.processor.generate_hash(f"{en_clean}{lg_clean}")
                            })
        elif isinstance(data, dict):
            # Handle dict format
            for key, value in data.items():
                if isinstance(value, dict):
                    en_text = value.get('english')
                    lg_text = value.get('luganda')
                elif isinstance(value, str):
                    # Assume key is English, value is Luganda
                    en_text = key
                    lg_text = value
                
                if en_text and lg_text:
                    en_clean = self.processor.clean_text(str(en_text))
                    lg_clean = self.processor.clean_text(str(lg_text))
                    
                    if self._is_valid_translation(en_clean, lg_clean):
                        translations.append({
                            'english': en_clean,
                            'luganda': lg_clean,
                            'source': f"audio_{path.name}",
                            'data_type': DataType.TRANSLATION.value,
                            'hash': self.processor.generate_hash(f"{en_clean}{lg_clean}")
                        })
        
        return translations
    
    def _read_audio_text_file(self, path: Path) -> List[Dict]:
        """Read audio translations from text/CSV file."""
        if path.suffix == '.csv':
            df = pd.read_csv(path, encoding='utf-8')
        else:  # .txt
            # Try different separators
            for sep in ['\t', '|', ',']:
                try:
                    df = pd.read_csv(path, sep=sep, encoding='utf-8')
                    break
                except Exception:
                    continue
            else:
                df = pd.DataFrame()
        
        translations = []
        if not df.empty:
            df = self._standardize_ttt_columns(df)
            
            for _, row in df.iterrows():
                if 'english' in row and 'luganda' in row:
                    en_text = self.processor.clean_text(str(row['english']))
                    lg_text = self.processor.clean_text(str(row['luganda']))
                    
                    if self._is_valid_translation(en_text, lg_text):
                        translations.append({
                            'english': en_text,
                            'luganda': lg_text,
                            'source': f"audio_{path.name}",
                            'data_type': DataType.TRANSLATION.value,
                            'hash': self.processor.generate_hash(f"{en_text}{lg_text}")
                        })
        
        return translations
    
    def _is_valid_translation(self, en_text: str, lg_text: str) -> bool:
        """Validate translation pair quality."""
        if not en_text or not lg_text:
            return False
        
        if len(en_text) < 5 or len(lg_text) < 5:
            return False
        
        # Check for common issues
        if en_text.strip().lower() == lg_text.strip().lower():
            return False  # Same text in both languages
        
        if len(en_text.split()) < 2 and len(lg_text.split()) < 2:
            return False  # Too short
        
        # Check for placeholder text
        placeholders = ['xxx', '???', '...', 'n/a', 'na', 'tbd', 'null', 'none']
        if any(ph in en_text.lower() for ph in placeholders):
            return False
        
        return True
    
    def _verify_translation_quality(self, df: pd.DataFrame):
        """Verify and log translation quality metrics."""
        self.logger.info("🔍 Verifying translation quality...")
        
        # Calculate basic statistics
        en_lengths = df['english'].apply(lambda x: len(x.split())).tolist()
        lg_lengths = df['luganda'].apply(lambda x: len(x.split())).tolist()
        
        avg_en_words = sum(en_lengths) / len(en_lengths) if en_lengths else 0
        avg_lg_words = sum(lg_lengths) / len(lg_lengths) if lg_lengths else 0
        
        # Check for ratio consistency (English to Luganda word ratio)
        ratios = []
        for en_len, lg_len in zip(en_lengths, lg_lengths):
            if en_len > 0:
                ratios.append(lg_len / en_len)
        
        avg_ratio = sum(ratios) / len(ratios) if ratios else 0
        
        self.logger.info(f"  ✓ Average English words: {avg_en_words:.1f}")
        self.logger.info(f"  ✓ Average Luganda words: {avg_lg_words:.1f}")
        self.logger.info(f"  ✓ Avg EN:LG word ratio: {avg_ratio:.2f}")
        
        # Check for source distribution
        sources = df['source'].value_counts()
        self.logger.info(f"  ✓ Sources: {len(sources)} unique sources")
        for source, count in sources.head(3).items():  # Top 3 sources
            self.logger.info(f"    - {source}: {count} pairs")

# Initialize data loader
data_loader = EnhancedKaggleDataLoader(config, logger)

## Data Validation

Uses [Pandera](https://pandera.readthedocs.io/) schemas to validate loaded DataFrames. Invalid rows are logged and dropped rather than silently passed through.

In [ ]:
# Data Validation with Pandera (expanded schema for governance)
try:
    import pandera as pa
    from pandera import Column, Check, DataFrameSchema
    PANDERA_AVAILABLE = True
except ImportError:
    PANDERA_AVAILABLE = False
    logger.warning("pandera not installed, skipping schema validation")

if PANDERA_AVAILABLE:
    faq_schema = DataFrameSchema({
        "question": Column(str, [
            Check.str_length(min_value=10, error="Question too short"),
            Check(lambda s: s.str.split().str.len() >= 3, error="Question needs >= 3 words"),
        ], nullable=False, coerce=True),
        "answer": Column(str, [
            Check.str_length(min_value=10, error="Answer too short"),
            Check(lambda s: s.str.split().str.len() >= 5, error="Answer needs >= 5 words"),
        ], nullable=False, coerce=True),
        "source": Column(str, nullable=False, coerce=True),
        "category": Column(str, nullable=True, coerce=True),
        "data_type": Column(str, nullable=True, coerce=True),
        "hash": Column(str, nullable=True, coerce=True),
    }, coerce=True)

    translation_schema = DataFrameSchema({
        "english": Column(str, [
            Check.str_length(min_value=5, error="English text too short"),
        ], nullable=False, coerce=True),
        "luganda": Column(str, [
            Check.str_length(min_value=5, error="Luganda text too short"),
        ], nullable=False, coerce=True),
    }, coerce=True)

    def validate_dataframe(df: pd.DataFrame, schema: DataFrameSchema, name: str) -> pd.DataFrame:
        """Validate a DataFrame against a schema, dropping invalid rows."""
        try:
            return schema.validate(df, lazy=True)
        except pa.errors.SchemaErrors as err:
            n_failures = len(err.failure_cases)
            logger.warning(f"Validation '{name}': {n_failures} failures detected, dropping invalid rows")
            invalid_indices = err.failure_cases["index"].unique()
            valid_df = df.drop(index=invalid_indices, errors="ignore").reset_index(drop=True)
            logger.info(f"  Kept {len(valid_df)}/{len(df)} rows after validation")
            return valid_df
else:
    def validate_dataframe(df: pd.DataFrame, schema, name: str) -> pd.DataFrame:
        return df
    faq_schema = None
    translation_schema = None

logger.info("✅ Data validation configured (expanded schema)")

## Pipeline Checkpoint & Progress Tracker

- **PipelineCheckpoint**: Saves/loads intermediate results so the pipeline can resume from the last successful stage after a failure
- **PipelineTracker**: Tracks timing and metrics for each pipeline stage, producing a structured summary at the end

In [ ]:
# Pipeline Checkpoint & Progress Tracker (JSONL/Parquet format with lineage)
import pyarrow.parquet as pq
import pyarrow as pa

@dataclass
class PipelineCheckpoint:
    """Save and load pipeline checkpoints using auditable formats (JSONL/Parquet)."""
    checkpoint_dir: Path = field(default_factory=lambda: config.output_dir / "checkpoints")

    def __post_init__(self):
        self.checkpoint_dir.mkdir(parents=True, exist_ok=True)

    def _add_lineage(self, data: Any, stage: str) -> Any:
        """Attach lineage metadata to checkpoint data."""
        lineage = {
            '_checkpoint_stage': stage,
            '_checkpoint_timestamp': datetime.now(timezone.utc).isoformat(),
            '_pipeline_version': config.dataset_version,
            '_random_seed': config.random_seed,
        }
        if isinstance(data, list) and data and isinstance(data[0], dict):
            for item in data:
                item.update(lineage)
        return data

    def save(self, stage: str, data: Any) -> None:
        """Save data for a pipeline stage using JSONL (lists) or Parquet (DataFrames)."""
        if isinstance(data, pd.DataFrame):
            path = self.checkpoint_dir / f"{stage}.parquet"
            data.to_parquet(path, index=False)
            logger.info(f"Checkpoint saved (Parquet): {stage} -> {path} ({len(data)} rows)")
        elif isinstance(data, (list, tuple)):
            # For tuples of mixed types (like load_data stage), use individual saves
            if isinstance(data, tuple):
                for i, item in enumerate(data):
                    self.save(f"{stage}_part{i}", item)
                # Save a manifest
                manifest = {'parts': len(data), 'stage': stage}
                manifest_path = self.checkpoint_dir / f"{stage}_manifest.json"
                with open(manifest_path, 'w') as f:
                    json.dump(manifest, f)
                return
            
            # List of dicts → JSONL
            path = self.checkpoint_dir / f"{stage}.jsonl"
            data_with_lineage = self._add_lineage(data.copy() if isinstance(data, list) else data, stage)
            with open(path, 'w', encoding='utf-8') as f:
                for item in data_with_lineage:
                    f.write(json.dumps(item, ensure_ascii=False, default=str) + '\n')
            logger.info(f"Checkpoint saved (JSONL): {stage} -> {path} ({len(data)} items)")
        else:
            # Fallback: JSON for simple types
            path = self.checkpoint_dir / f"{stage}.json"
            with open(path, 'w', encoding='utf-8') as f:
                json.dump(data, f, ensure_ascii=False, default=str)
            logger.info(f"Checkpoint saved (JSON): {stage} -> {path}")

    def load(self, stage: str) -> Optional[Any]:
        """Load data for a pipeline stage."""
        # Check for manifest (tuple data)
        manifest_path = self.checkpoint_dir / f"{stage}_manifest.json"
        if manifest_path.exists():
            with open(manifest_path) as f:
                manifest = json.load(f)
            parts = []
            for i in range(manifest['parts']):
                part = self.load(f"{stage}_part{i}")
                parts.append(part)
            logger.info(f"Checkpoint loaded (multi-part): {stage}")
            return tuple(parts)
        
        # Parquet
        path = self.checkpoint_dir / f"{stage}.parquet"
        if path.exists():
            df = pd.read_parquet(path)
            logger.info(f"Checkpoint loaded (Parquet): {stage} ({len(df)} rows)")
            return df
        
        # JSONL
        path = self.checkpoint_dir / f"{stage}.jsonl"
        if path.exists():
            data = []
            with open(path, 'r', encoding='utf-8') as f:
                for line in f:
                    if line.strip():
                        data.append(json.loads(line))
            # Strip lineage metadata
            for item in data:
                for key in list(item.keys()):
                    if key.startswith('_checkpoint_') or key.startswith('_pipeline_') or key.startswith('_random_'):
                        del item[key]
            logger.info(f"Checkpoint loaded (JSONL): {stage} ({len(data)} items)")
            return data
        
        # JSON
        path = self.checkpoint_dir / f"{stage}.json"
        if path.exists():
            with open(path) as f:
                data = json.load(f)
            logger.info(f"Checkpoint loaded (JSON): {stage}")
            return data
        
        # Legacy pickle fallback (read-only, for migration)
        path = self.checkpoint_dir / f"{stage}.pkl"
        if path.exists():
            import pickle
            with open(path, "rb") as f:
                data = pickle.load(f)
            logger.warning(f"Checkpoint loaded (legacy pickle): {stage} - consider re-running pipeline")
            return data
        
        return None

    def has(self, stage: str) -> bool:
        exts = ['.parquet', '.jsonl', '.json', '.pkl', '_manifest.json']
        return any((self.checkpoint_dir / f"{stage}{ext}").exists() for ext in exts)

    def clear(self) -> None:
        for f in self.checkpoint_dir.glob("*"):
            if f.is_file():
                f.unlink()
        logger.info("All checkpoints cleared")


class PipelineTracker:
    """Track pipeline stages with timing and metrics."""

    def __init__(self):
        self.stages: Dict[str, Dict[str, Any]] = {}
        self._current_stage: Optional[str] = None
        self._stage_start: float = 0.0

    @contextmanager
    def stage(self, name: str):
        self._current_stage = name
        self._stage_start = time.time()
        self.stages[name] = {"status": "running", "metrics": {}}
        logger.info(f"[STAGE] {name} started")
        try:
            yield
            elapsed = time.time() - self._stage_start
            self.stages[name]["status"] = "completed"
            self.stages[name]["elapsed_s"] = round(elapsed, 2)
            logger.info(f"[STAGE] {name} completed in {elapsed:.1f}s")
        except Exception as e:
            elapsed = time.time() - self._stage_start
            self.stages[name]["status"] = "failed"
            self.stages[name]["elapsed_s"] = round(elapsed, 2)
            self.stages[name]["error"] = str(e)
            logger.error(f"[STAGE] {name} failed after {elapsed:.1f}s: {e}")
            raise
        finally:
            self._current_stage = None

    def metric(self, key: str, value: Any) -> None:
        if self._current_stage and self._current_stage in self.stages:
            self.stages[self._current_stage]["metrics"][key] = value

    def summary(self) -> Dict[str, Any]:
        total_time = sum(s.get("elapsed_s", 0) for s in self.stages.values())
        return {"stages": self.stages, "total_time_s": round(total_time, 2)}


checkpoint = PipelineCheckpoint()
tracker = PipelineTracker()
logger.info("✅ Checkpoint system initialized (JSONL/Parquet with lineage)")

## Teacher Model QA Generation

`EnhancedTeacherModel` uses a 4-bit quantized LLM (Qwen2.5-7B-Instruct) to generate QA pairs from PDF text chunks. Features:

- **4-bit NF4 quantization** with double quantization for memory efficiency
- **Batched inference** with automatic GPU memory cleanup
- **Multi-pattern QA parsing** with fallback generation
- **Confidence scoring** based on answer relevance to source text

In [ ]:
# Cell 7: Enhanced Teacher Model with GeneratedQA class
class EnhancedTeacherModel:
    """Enhanced teacher model with better QA generation."""
    
    def __init__(self, model_name: str, config: KaggleConfig):
        self.model_name = model_name
        self.config = config
        self.pipeline = None
        self.tokenizer = None
        self.device_count = torch.cuda.device_count()
        self.logger = logging.getLogger(__name__)
        
        # Question types with weights
        self.question_types = {
            "factual": 0.4,      # Direct facts from text
            "inferential": 0.3,  # Requires inference
            "application": 0.2,  # Real-world application
            "procedural": 0.1    # Step-by-step processes
        }
    
    def load(self):
        """Load model with 4-bit quantization."""
        self.logger.info(f"🚀 Loading {self.model_name} on {self.device_count} GPUs...")
        
        # 4-bit quantization for memory efficiency
        quant_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True
        )
        
        # Load tokenizer
        self.tokenizer = AutoTokenizer.from_pretrained(
            self.model_name,
            cache_dir=str(self.config.cache_dir),
            trust_remote_code=True
        )
        
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
        self.tokenizer.padding_side = "left"
        
        # Load model
        try:
            model = AutoModelForCausalLM.from_pretrained(
                self.model_name,
                quantization_config=quant_config,
                device_map="auto",
                torch_dtype=torch.float16,
                cache_dir=str(self.config.cache_dir),
                trust_remote_code=True,
                low_cpu_mem_usage=True,
            )
        except Exception:
            # Fallback without flash attention
            model = AutoModelForCausalLM.from_pretrained(
                self.model_name,
                quantization_config=quant_config,
                device_map="auto",
                torch_dtype=torch.float16,
                cache_dir=str(self.config.cache_dir),
                trust_remote_code=True,
                low_cpu_mem_usage=True
            )
        
        # Create pipeline
        self.pipeline = pipeline(
            "text-generation",
            model=model,
            tokenizer=self.tokenizer,
            max_new_tokens=self.config.max_new_tokens,
            temperature=self.config.temperature,
            top_p=self.config.top_p,
            do_sample=True,
            repetition_penalty=1.1,
            pad_token_id=self.tokenizer.pad_token_id,
            eos_token_id=self.tokenizer.eos_token_id,
        )
        
        self.logger.info(f"✅ Model loaded successfully on {self.device_count} GPU(s)")
    
    @torch.inference_mode()
    def generate_enhanced_qa(self, chunks: List[Dict]) -> List[GeneratedQA]:
        """Generate enhanced QA pairs with better quality control."""
        self.logger.info(f" Generating enhanced QA from {len(chunks)} chunks...")
        
        all_qa_pairs = []
        
        # Process in batches
        batch_size = max(1, self.config.batch_size_per_gpu * self.device_count)
        
        with tqdm(total=len(chunks), desc="Generating QA") as pbar:
            for i in range(0, len(chunks), batch_size):
                batch_chunks = chunks[i:i+batch_size]
                batch_texts = [chunk["text"] for chunk in batch_chunks]
                
                try:
                    # Generate QA for this batch
                    batch_results = self._generate_batch_qa(batch_texts, batch_chunks)
                    
                    # Process and validate results
                    for chunk, qa_list in zip(batch_chunks, batch_results):
                        valid_qa = self._validate_and_score_qa(qa_list, chunk["text"])
                        
                        for qa in valid_qa:
                            generated_qa = GeneratedQA(
                                question=qa["question"],
                                answer=qa["answer"],
                                chunk_text=chunk["text"][:1000],  # Store more context
                                source_pdf=chunk["source"],
                                chunk_id=chunk["chunk_id"],
                                question_type=qa["type"],
                                confidence=qa.get("confidence", config.default_qa_confidence)
                            )
                            all_qa_pairs.append(generated_qa)
                        
                        pbar.update(1)
                    
                    # Clean GPU memory
                    if i % (batch_size * 3) == 0:
                        self.cleanup_gpu()
                        
                except Exception as e:
                    self.logger.error(f"Error processing batch {i}: {e}")
                    pbar.update(len(batch_chunks))
        
        self.logger.info(f"✅ Generated {len(all_qa_pairs)} enhanced QA pairs")
        return all_qa_pairs
    
    def _generate_batch_qa(self, texts: List[str], chunks: List[Dict]) -> List[List[Dict]]:
        """Generate QA pairs from batch of texts with enhanced prompts."""
        prompts = []
        
        for text, chunk in zip(texts, chunks):
            # Create enhanced prompt based on chunk content
            category = chunk.get("category", "general")
            prompt = self._create_enhanced_prompt(text, category)
            prompts.append(prompt)
        
        # Generate responses
        outputs = self.pipeline(
            prompts,
            batch_size=max(1, len(prompts) // 2),
            truncation=True,
            max_new_tokens=200  # Slightly more for better answers
        )
        
        # Parse outputs
        results = []
        for output, text, chunk in zip(outputs, texts, chunks):
            generated = output[0]["generated_text"]
            qa_pairs = self._parse_enhanced_output(generated, text, chunk)
            results.append(qa_pairs)
        
        return results
    
    def _create_enhanced_prompt(self, text: str, category: str) -> str:
        """Create enhanced prompt for QA generation."""
        # Extract key terms for better question generation
        key_terms = processor.extract_key_terms(text)
        
        prompt = f"""You are a URA tax expert creating training data for a tax chatbot.

Document Category: {category}
Document Excerpt: {text[:350]}...

Key Terms: {', '.join(key_terms[:5]) if key_terms else 'tax, URA, compliance'}

Generate {self.config.questions_per_chunk} diverse, high-quality question-answer pairs.
For each pair, include:
1. A clear, specific question that a taxpayer might ask
2. A detailed, accurate answer based on the text
3. Question type (factual, inferential, application, or procedural)
4. Confidence score (0.0 to 1.0) for accuracy

Format:
Q1: [question]
A1: [answer]
Type1: [question_type]
Confidence1: [confidence]

Q2: [question]
A2: [answer]
Type2: [question_type]
Confidence2: [confidence]

..."""
        
        return prompt
    
    def _parse_enhanced_output(self, generated: str, original_text: str, chunk: Dict) -> List[Dict]:
        """Parse enhanced output with better validation."""
        qa_pairs = []
        
        # Pattern for enhanced format
        patterns = [
            # Enhanced format with confidence
            r'Q\d+:\s*(.*?)\s*A\d+:\s*(.*?)\s*Type\d+:\s*(\w+)\s*Confidence\d+:\s*([0-9.]+)',
            # Basic format
            r'Q:\s*(.*?)\s*A:\s*(.*?)(?=\s*(?:Q:|Type:|$))',
            # Numbered format
            r'\d+\.\s*Question:\s*(.*?)\s*Answer:\s*(.*?)(?=\s*\d+\.|$)'
        ]
        
        for pattern in patterns:
            matches = re.findall(pattern, generated, re.DOTALL | re.IGNORECASE)
            
            for match in matches:
                if len(match) >= 2:
                    question = match[0].strip()
                    answer = match[1].strip()
                    
                    # Extract additional metadata if available
                    q_type = "factual"
                    confidence = config.default_qa_confidence
                    
                    if len(match) >= 4:
                        q_type = match[2].strip().lower()
                        try:
                            confidence = float(match[3].strip())
                        except (ValueError, TypeError):
                            confidence = config.default_qa_confidence
                    
                    # Validate QA pair
                    is_valid, reason = processor.validate_qa_pair(
                        question, answer,
                        self.config.min_question_words,
                        self.config.min_answer_words
                    )
                    
                    if is_valid:
                        # Ensure answer relates to original text
                        similarity = processor.calculate_similarity(answer, original_text[:500])
                        if similarity > config.qa_relevance_threshold:  # Minimum relevance
                            qa_pairs.append({
                                "question": question,
                                "answer": answer,
                                "type": q_type,
                                "confidence": confidence,
                                "similarity": similarity
                            })
        
        # If no valid pairs found, create simple ones
        if not qa_pairs:
            qa_pairs = self._create_fallback_qa(original_text, chunk)
        
        return qa_pairs[:self.config.questions_per_chunk]
    
    def _create_fallback_qa(self, text: str, chunk: Dict) -> List[Dict]:
        """Create fallback QA pairs when generation fails."""
        qa_pairs = []
        
        # Extract sentences for QA generation
        sentences = re.split(r'[.!?]', text)
        meaningful_sentences = [s.strip() for s in sentences if len(s.split()) > 5]
        
        for i, sentence in enumerate(meaningful_sentences[:self.config.questions_per_chunk]):
            if len(sentence) > 20:
                # Create question from sentence
                words = sentence.split()
                if len(words) > 8:
                    # Use first part of sentence as question basis
                    topic = ' '.join(words[:4]).lower()
                    question = f"What does the URA say about {topic}?"
                else:
                    question = f"Explain this URA guideline: {sentence[:100]}..."
                
                qa_pairs.append({
                    "question": question,
                    "answer": sentence,
                    "type": "factual",
                    "confidence": config.fallback_qa_confidence,
                    "similarity": 1.0
                })
        
        return qa_pairs
    
    def _validate_and_score_qa(self, qa_list: List[Dict], original_text: str) -> List[Dict]:
        """Validate and score generated QA pairs."""
        valid_qa = []
        
        for qa in qa_list:
            # Basic validation
            is_valid, reason = processor.validate_qa_pair(
                qa["question"], qa["answer"],
                self.config.min_question_words,
                self.config.min_answer_words
            )
            
            if is_valid:
                # Calculate relevance score
                relevance = processor.calculate_similarity(qa["answer"], original_text[:500])
                
                # Adjust confidence based on relevance
                adjusted_confidence = qa.get("confidence", config.default_qa_confidence) * (0.5 + 0.5 * relevance)
                
                # Ensure minimum confidence
                if adjusted_confidence > config.min_qa_confidence:
                    qa["confidence"] = adjusted_confidence
                    valid_qa.append(qa)
        
        # Sort by confidence and limit
        valid_qa.sort(key=lambda x: x.get("confidence", 0), reverse=True)
        return valid_qa[:self.config.questions_per_chunk]
    
    def cleanup_gpu(self):
        """Clean up GPU memory."""
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.synchronize()
            torch.cuda.reset_peak_memory_stats()
    
    def unload(self):
        """Unload model from GPU."""
        del self.pipeline
        del self.tokenizer
        self.cleanup_gpu()
        self.logger.info("✅ Model unloaded from GPU")

# Generation function
def generate_enhanced_teacher_qa(pdf_chunks: List[Dict], config: KaggleConfig) -> List[Dict]:
    """Generate enhanced QA pairs from PDF chunks."""
    logger.info(f" Generating enhanced teacher QA from {len(pdf_chunks)} PDF chunks...")
    
    if not pdf_chunks:
        logger.warning("No PDF chunks to process")
        return []
    
    # Initialize enhanced teacher model
    teacher = EnhancedTeacherModel(config.teacher_model, config)
    
    try:
        teacher.load()
        
        # Generate enhanced QA pairs
        generated_qa_objects = teacher.generate_enhanced_qa(pdf_chunks)
        
        # Convert to dictionaries
        all_qa_pairs = []
        for qa_obj in generated_qa_objects:
            qa_dict = qa_obj.to_dict()
            qa_dict.update({
                "data_type": DataType.TEACHER_QA.value,
                "hash": processor.generate_hash(f"{qa_obj.question}{qa_obj.answer}")
            })
            all_qa_pairs.append(qa_dict)
        
        logger.info(f"✅ Generated {len(all_qa_pairs)} enhanced teacher QA pairs")
        
        # Save to file
        teacher_qa_file = config.output_dir / "enhanced_teacher_qa.jsonl"
        with open(teacher_qa_file, 'w', encoding='utf-8') as f:
            for qa in all_qa_pairs:
                f.write(json.dumps(qa, ensure_ascii=False) + '\n')
        
        logger.info(f"💾 Saved enhanced teacher QA to {teacher_qa_file}")
        
        return all_qa_pairs
        
    except Exception as e:
        logger.error(f"❌ Error generating enhanced teacher QA: {e}", exc_info=True)
        return []
    finally:
        teacher.unload()

## Data Augmentation

`EnhancedDataAugmentor` creates multiple training format variations from each data point:

| Format | Description |
|--------|-------------|
| **Instruction** | `<instruction>` / `<input>` / `<output>` tags |
| **Chat** | System / user / assistant message structure |
| **Gemma** | `<start_of_turn>` / `<end_of_turn>` markers |

Translation pairs get 5 format variations including bilingual and bidirectional formats.

In [ ]:
# Cell 8: Enhanced Data Augmentor with Translation Support
class EnhancedDataAugmentor:
    """Enhanced augmentor with translation support."""
    
    def __init__(self, config: KaggleConfig, logger: logging.Logger):
        self.config = config
        self.logger = logger
        self.processor = TextProcessor()
        
        # Templates
        self.instruction_templates = [
            "Answer this tax question: {question}",
            "As a URA customer service assistant, answer: {question}",
            "Help me understand: {question}",
            "Tax query: {question}",
            "URA FAQ: {question}",
            "{question}",
        ]
        
        self.system_prompts = [
            "You are a helpful URA (Uganda Revenue Authority) customer service assistant. Answer tax-related questions accurately and concisely.",
            "You are an expert on Ugandan tax laws and URA procedures. Provide clear, accurate answers.",
        ]
    
    def augment(self, faq_df, teacher_qa, translations_df):
        """Create augmented training data including translations."""
        self.logger.info("🔄 Creating augmented training data with translations...")
        
        all_data = []
        seen_hashes = set()
        
        # Process FAQ data
        if not faq_df.empty:
            all_data.extend(self._process_faq_data(faq_df, seen_hashes))
        
        # Process teacher QA
        if teacher_qa:
            all_data.extend(self._process_teacher_qa(teacher_qa, seen_hashes))
        
        # Process translations
        if not translations_df.empty and self.config.support_luganda:
            self.logger.info(f"🌍 Processing {len(translations_df)} translation pairs")
            all_data.extend(self._augment_translation_data(translations_df, seen_hashes))
        
        # Phase 2: Post-augmentation semantic dedup (MinHash near-duplicate detection)
        unique_data = self._semantic_dedup(all_data)
        
        # Phase 3: PII redaction before export
        if self.config.redact_pii:
            for item in unique_data:
                for text_key in ('question', 'answer', 'instruction', 'input', 'output'):
                    if text_key in item and item[text_key]:
                        item[text_key] = self.processor.redact_pii(item[text_key], self.config.pii_patterns)
        
        self.logger.info(f"✅ Created {len(unique_data)} unique training samples (from {len(all_data)} pre-dedup)")
        
        # Log composition
        self._log_data_composition(unique_data)
        
        return unique_data
    
    def _semantic_dedup(self, data: List[Dict]) -> List[Dict]:
        """Remove near-duplicates using MinHash LSH (fuzzy matching).
        
        Phase 1 (exact hash) is already done in _process_* methods.
        Phase 2 (semantic) catches paraphrases and near-duplicates.
        """
        if not data:
            return data
        
        if not MINHASH_AVAILABLE or len(data) < 10:
            self.logger.info(f"  Skipping semantic dedup (MINHASH_AVAILABLE={MINHASH_AVAILABLE}, n={len(data)})")
            return data
        
        self.logger.info(f"  Running MinHash semantic dedup on {len(data)} items (threshold={self.config.minhash_threshold})...")
        
        lsh = MinHashLSH(threshold=self.config.minhash_threshold, num_perm=self.config.minhash_num_perm)
        unique = []
        n_dropped = 0
        
        for idx, item in enumerate(data):
            text = (item.get('question', '') or item.get('instruction', '')) + ' ' + (item.get('answer', '') or item.get('output', ''))
            text = text.lower().strip()
            
            # Create MinHash
            m = MinHash(num_perm=self.config.minhash_num_perm)
            for word in text.split():
                m.update(word.encode('utf-8'))
            
            # Check for near-duplicates
            try:
                result = lsh.query(m)
                if result:
                    n_dropped += 1
                    continue
                lsh.insert(f"item_{idx}", m)
                unique.append(item)
            except Exception:
                unique.append(item)
        
        self.logger.info(f"  Semantic dedup: {len(data)} → {len(unique)} (dropped {n_dropped} near-duplicates)")
        return unique
    
    def _process_faq_data(self, faq_df, seen_hashes):
        """Process FAQ data into training formats."""
        data = []
        
        for _, row in faq_df.iterrows():
            # Create multiple formats
            formats = self._create_formats(
                question=row['question'],
                answer=row['answer'],
                category=row.get('category', ''),
                source=row['source'],
                data_type=DataType.FAQ.value
            )
            
            for fmt in formats:
                item_hash = self.processor.generate_hash(
                    f"{fmt.get('question', fmt.get('instruction', ''))}"
                    f"{fmt.get('answer', fmt.get('output', ''))}"
                )
                
                if item_hash not in seen_hashes:
                    fmt['hash'] = item_hash
                    data.append(fmt)
                    seen_hashes.add(item_hash)
        
        return data
    
    def _process_teacher_qa(self, teacher_qa, seen_hashes):
        """Process teacher-generated QA."""
        data = []
        
        for qa in teacher_qa:
            formats = self._create_formats(
                question=qa['question'],
                answer=qa['answer'],
                category=qa.get('category', 'teacher_generated'),
                source=qa['source'],
                data_type=DataType.TEACHER_QA.value,
                confidence=qa.get('confidence', config.default_qa_confidence)
            )
            
            for fmt in formats:
                item_hash = self.processor.generate_hash(
                    f"{fmt.get('question', fmt.get('instruction', ''))}"
                    f"{fmt.get('answer', fmt.get('output', ''))}"
                )
                
                if item_hash not in seen_hashes:
                    fmt['hash'] = item_hash
                    fmt['confidence'] = qa.get('confidence', config.default_qa_confidence)
                    data.append(fmt)
                    seen_hashes.add(item_hash)
        
        return data
    
    def _augment_translation_data(self, translations_df: pd.DataFrame, seen_hashes: set) -> List[Dict[str, Any]]:
        """Augment translation data with multiple formats."""
        augmented = []
        
        for _, row in translations_df.iterrows():
            formats = self._create_luganda_formats(row.to_dict())
            
            for fmt in formats:
                item_hash = self.processor.generate_hash(
                    f"{fmt.get('question', fmt.get('instruction', ''))}"
                    f"{fmt.get('answer', fmt.get('output', ''))}"
                )
                
                if item_hash not in seen_hashes:
                    fmt['hash'] = item_hash
                    augmented.append(fmt)
                    seen_hashes.add(item_hash)
        
        return augmented
    
    def _create_luganda_formats(self, translation: Dict[str, Any]) -> List[Dict[str, Any]]:
        """Create training pairs with Luganda translation."""
        en_text = translation['english']
        lg_text = translation['luganda']
        source = translation['source']
        
        formats = []
        
        # Format 1: English to Luganda translation
        formats.append({
            'instruction': f"Translate this English text to Luganda: {en_text}",
            'input': '',
            'output': lg_text,
            'category': 'translation_en_to_lg',
            'source': source,
            'data_type': DataType.TRANSLATION.value,
            'format': FormatType.INSTRUCTION.value,
            'timestamp': datetime.now().isoformat()
        })
        
        # Format 2: Luganda to English translation
        formats.append({
            'instruction': f"Translate this Luganda text to English: {lg_text}",
            'input': '',
            'output': en_text,
            'category': 'translation_lg_to_en',
            'source': source,
            'data_type': DataType.TRANSLATION.value,
            'format': FormatType.INSTRUCTION.value,
            'timestamp': datetime.now().isoformat()
        })
        
        # Format 3: Bilingual response
        if self.config.create_bilingual_pairs:
            formats.append({
                'instruction': f"Provide information about this in both English and Luganda: {en_text[:50]}",
                'input': '',
                'output': f"English: {en_text}\n\nLuganda: {lg_text}",
                'category': 'bilingual',
                'source': source,
                'data_type': DataType.TRANSLATION.value,
                'format': FormatType.INSTRUCTION.value,
                'timestamp': datetime.now().isoformat()
            })
        
        # Format 4: Gemma format for English-Luganda
        formats.append({
            'text': f"<start_of_turn>user\nTranslate to Luganda: {en_text}<end_of_turn>\n<start_of_turn>model\n{lg_text}<end_of_turn>",
            'question': f"Translate to Luganda: {en_text}",
            'answer': lg_text,
            'category': 'translation',
            'source': source,
            'data_type': DataType.TRANSLATION.value,
            'format': FormatType.GEMMA.value,
            'timestamp': datetime.now().isoformat()
        })
        
        # Format 5: Chat format for translation
        formats.append({
            'messages': [
                {'role': 'system', 'content': 'You are a bilingual assistant fluent in English and Luganda.'},
                {'role': 'user', 'content': f"Translate this to Luganda: {en_text}"},
                {'role': 'assistant', 'content': lg_text}
            ],
            'category': 'translation_chat',
            'source': source,
            'data_type': DataType.TRANSLATION.value,
            'format': FormatType.CHAT.value,
            'timestamp': datetime.now().isoformat()
        })
        
        return formats
    
    def _create_formats(self, question, answer, category, source, data_type, confidence=1.0):
        """Create multiple format variations."""
        formats = []
        
        # Instruction format
        instruction = random.choice(self.instruction_templates).format(question=question)
        if category:
            instruction = f"[{category.upper()}] {instruction}"
        
        formats.append({
            'instruction': instruction,
            'input': '',
            'output': answer,
            'category': category,
            'source': source,
            'data_type': data_type,
            'confidence': confidence,
            'format': FormatType.INSTRUCTION.value,
            'timestamp': datetime.now().isoformat()
        })
        
        # Chat format
        system_prompt = random.choice(self.system_prompts)
        if category:
            system_prompt = f"{system_prompt} Specializing in {category}."
        
        formats.append({
            'messages': [
                {'role': 'system', 'content': system_prompt},
                {'role': 'user', 'content': question},
                {'role': 'assistant', 'content': answer}
            ],
            'category': category,
            'source': source,
            'data_type': data_type,
            'confidence': confidence,
            'format': FormatType.CHAT.value,
            'timestamp': datetime.now().isoformat()
        })
        
        # Gemma format
        formats.append({
            'text': f"<start_of_turn>user\n{question}<end_of_turn>\n<start_of_turn>model\n{answer}<end_of_turn>",
            'question': question,
            'answer': answer,
            'category': category,
            'source': source,
            'data_type': data_type,
            'confidence': confidence,
            'format': FormatType.GEMMA.value,
            'timestamp': datetime.now().isoformat()
        })
        
        return formats
    
    def _log_data_composition(self, data: List[Dict]):
        """Log detailed data composition."""
        counts_by_type = {}
        counts_by_format = {}
        counts_by_source = {}
        
        for item in data:
            # By data type
            dtype = item.get('data_type', 'unknown')
            counts_by_type[dtype] = counts_by_type.get(dtype, 0) + 1
            
            # By format
            fmt = item.get('format', 'unknown')
            counts_by_format[fmt] = counts_by_format.get(fmt, 0) + 1
            
            # By source
            src = item.get('source', 'unknown')
            if isinstance(src, str):
                # Group similar sources
                if 'ttt_' in src:
                    src_group = 'ttt'
                elif 'audio_' in src:
                    src_group = 'audio'
                elif 'teacher' in src.lower():
                    src_group = 'teacher'
                elif 'faq' in src.lower():
                    src_group = 'faq'
                else:
                    src_group = src.split('_')[0] if '_' in src else src
                
                counts_by_source[src_group] = counts_by_source.get(src_group, 0) + 1
        
        self.logger.info("\n📊 DATA COMPOSITION:")
        self.logger.info("  By Data Type:")
        for dtype, count in counts_by_type.items():
            self.logger.info(f"    {dtype}: {count}")
        
        self.logger.info("\n  By Format:")
        for fmt, count in counts_by_format.items():
            self.logger.info(f"    {fmt}: {count}")
        
        self.logger.info("\n  By Source Group:")
        for src, count in counts_by_source.items():
            self.logger.info(f"    {src}: {count}")

# Initialize augmentor
augmentor = EnhancedDataAugmentor(config, logger)

In [ ]:
# Cell 9: Data Exporter
class DataExporter:
    """Export data in multiple formats."""
    
    def __init__(self, config: KaggleConfig, logger: logging.Logger):
        self.config = config
        self.logger = logger
    
    def export_jsonl(self, data, output_path):
        """Export data to JSONL format."""
        output_path.parent.mkdir(parents=True, exist_ok=True)
        
        with open(output_path, 'w', encoding='utf-8') as f:
            for item in data:
                f.write(json.dumps(item, ensure_ascii=False) + '\n')
        
        self.logger.info(f"💾 Exported {len(data)} samples to {output_path}")
    
    def export_train_val_split(self, data, output_dir):
        """Export with stratified train/validation split (grouped by source to prevent leakage)."""
        if not data:
            self.logger.warning("No data to split")
            return
        
        # Group by source to prevent leakage (variants of same source stay together)
        source_groups = {}
        for item in data:
            source = item.get('source', 'unknown')
            # Group by base source (strip augmentation suffixes)
            base_source = re.sub(r'_augmented.*|_variation.*', '', source)
            source_groups.setdefault(base_source, []).append(item)
        
        # Stratified split: each source contributes proportionally to train/val
        train_data, val_data = [], []
        
        for source, items in source_groups.items():
            random.shuffle(items)
            n_val = max(1, int(len(items) * self.config.train_val_split))
            val_data.extend(items[:n_val])
            train_data.extend(items[n_val:])
        
        # Final shuffle within splits (preserving group separation)
        random.shuffle(train_data)
        random.shuffle(val_data)
        
        splits_dir = output_dir / 'splits'
        splits_dir.mkdir(parents=True, exist_ok=True)
        
        train_path = splits_dir / 'train.jsonl'
        val_path = splits_dir / 'val.jsonl'
        
        self.export_jsonl(train_data, train_path)
        self.export_jsonl(val_data, val_path)
        
        # Log split composition
        train_sources = set(item.get('source', '?') for item in train_data)
        val_sources = set(item.get('source', '?') for item in val_data)
        leakage = train_sources & val_sources
        
        self.logger.info(f"📊 Stratified Train/Val split: {len(train_data)} train, {len(val_data)} val")
        self.logger.info(f"   Train sources: {len(train_sources)}, Val sources: {len(val_sources)}")
        if leakage:
            self.logger.info(f"   Source overlap (expected for multi-item sources): {len(leakage)}")
    
    def export_statistics(self, data, output_path):
        """Export dataset statistics."""
        stats = {
            "total_samples": len(data),
            "by_format": {},
            "by_category": {},
            "by_data_type": {},
            "quality_metrics": {
                "avg_question_length": 0,
                "avg_answer_length": 0,
            }
        }
        
        question_lengths = []
        answer_lengths = []
        
        for item in data:
            # Format statistics
            fmt = item.get('format', 'unknown')
            stats["by_format"][fmt] = stats["by_format"].get(fmt, 0) + 1
            
            # Category statistics
            cat = item.get('category', 'unknown')
            stats["by_category"][cat] = stats["by_category"].get(cat, 0) + 1
            
            # Data type statistics
            dtype = item.get('data_type', 'unknown')
            stats["by_data_type"][dtype] = stats["by_data_type"].get(dtype, 0) + 1
            
            # Quality metrics
            question = item.get('question') or item.get('instruction', '')
            answer = item.get('answer') or item.get('output', '')
            
            if question:
                question_lengths.append(len(question.split()))
            if answer:
                answer_lengths.append(len(answer.split()))
        
        if question_lengths:
            stats["quality_metrics"]["avg_question_length"] = sum(question_lengths) / len(question_lengths)
        if answer_lengths:
            stats["quality_metrics"]["avg_answer_length"] = sum(answer_lengths) / len(answer_lengths)
        
        with open(output_path, 'w', encoding='utf-8') as f:
            json.dump(stats, f, indent=2, ensure_ascii=False)
        
        self.logger.info(f"📈 Statistics exported to {output_path}")
        
        # Print summary
        self.logger.info("\n" + "="*50)
        self.logger.info("DATASET STATISTICS")
        self.logger.info("="*50)
        self.logger.info(f"Total samples: {stats['total_samples']}")
        for fmt, count in stats['by_format'].items():
            self.logger.info(f"  {fmt}: {count}")
        for dtype, count in stats['by_data_type'].items():
            self.logger.info(f"  {dtype}: {count}")
        
        return stats

# Initialize exporter
exporter = DataExporter(config, logger)

## Data Provenance & Integrity Verification

Supply-chain security for training data:
- **Source trust policy**: Only allow data from verified Kaggle slugs
- **Content integrity**: SHA-256 checksums for all source files
- **Manifest generation**: Auditable record of all data lineage

In [ ]:
# Data Provenance & Integrity Verification
class DataProvenanceVerifier:
    """Verify data integrity and generate provenance manifests."""
    
    def __init__(self, config: KaggleConfig, logger: logging.Logger):
        self.config = config
        self.logger = logger
    
    def verify_source_trust(self, dataset_slug: str) -> bool:
        """Check if a dataset slug is in the trusted sources list."""
        normalized = dataset_slug.lower().strip()
        for trusted in self.config.trusted_sources:
            if normalized == trusted.lower() or normalized.endswith(trusted.lower()):
                self.logger.info(f"✅ Source trusted: {dataset_slug}")
                return True
        self.logger.warning(f"⚠️ UNTRUSTED source: {dataset_slug}")
        return False
    
    def compute_file_hashes(self, directory: Path) -> Dict[str, str]:
        """Compute SHA-256 hashes for all files in a directory."""
        hashes = {}
        if not directory.exists():
            return hashes
        for file_path in sorted(directory.rglob("*")):
            if file_path.is_file() and file_path.stat().st_size < 500_000_000:  # Skip >500MB
                sha = hashlib.sha256()
                with open(file_path, "rb") as f:
                    for chunk in iter(lambda: f.read(8192), b""):
                        sha.update(chunk)
                hashes[str(file_path.relative_to(directory))] = sha.hexdigest()
        return hashes
    
    def generate_manifest(self, data_dir: Path, dataset_slug: str, 
                          faq_count: int, pdf_count: int, translation_count: int) -> Dict:
        """Generate a signed provenance manifest for the dataset."""
        file_hashes = self.compute_file_hashes(data_dir)
        
        manifest = {
            "manifest_version": "1.0",
            "generated_at": datetime.now(timezone.utc).isoformat(),
            "pipeline_version": self.config.dataset_version,
            "random_seed": self.config.random_seed,
            "dataset_source": {
                "slug": dataset_slug,
                "trusted": self.verify_source_trust(dataset_slug),
            },
            "data_counts": {
                "faq_pairs": faq_count,
                "pdf_chunks": pdf_count,
                "translation_pairs": translation_count,
            },
            "file_checksums": file_hashes,
            "checksum_algorithm": "sha256",
            "integrity_hash": hashlib.sha256(
                json.dumps(file_hashes, sort_keys=True).encode()
            ).hexdigest(),
        }
        
        # Save manifest
        manifest_path = self.config.output_dir / "data_manifest.json"
        with open(manifest_path, 'w') as f:
            json.dump(manifest, f, indent=2)
        
        self.logger.info(f"📋 Provenance manifest: {manifest_path}")
        self.logger.info(f"   Files hashed: {len(file_hashes)}")
        self.logger.info(f"   Integrity hash: {manifest['integrity_hash'][:16]}...")
        
        return manifest

provenance = DataProvenanceVerifier(config, logger)
logger.info("✅ Provenance verifier initialized")

## Synthetic QA Quality Gates

Enhanced quality checks for teacher-generated QA:
- **Groundedness**: Does the answer reference content from the source chunk?
- **Citation presence**: Does the answer mention the topic from the question?
- **Length/format guards**: Reject too-short or malformed QA pairs
- **Confidence-based reject sampling**: Filter below threshold

In [ ]:
# QA Quality Gates (groundedness + citation + reject-sampling)
class QAQualityGate:
    """Enhanced quality checks for synthetic QA pairs."""
    
    def __init__(self, config: KaggleConfig, logger: logging.Logger):
        self.config = config
        self.logger = logger
        self.processor = TextProcessor()
    
    def evaluate_qa_pair(self, question: str, answer: str, source_chunk: str) -> Dict[str, Any]:
        """Score a QA pair for quality across multiple dimensions."""
        scores = {}
        
        # 1. Length checks
        q_words = len(question.split())
        a_words = len(answer.split())
        scores['q_length_ok'] = q_words >= self.config.min_question_words
        scores['a_length_ok'] = a_words >= self.config.min_answer_words
        
        # 2. Groundedness: how much of the answer is grounded in the source chunk
        if source_chunk:
            chunk_words = set(source_chunk.lower().split())
            answer_words = set(answer.lower().split())
            stopwords = {'the','a','an','is','are','was','were','be','in','on','at','to','for','of','and','or','but','with','by','from','as','it','this','that','can','will','would','should','do','does','did','has','have','had','i','you','we','they'}
            chunk_content = chunk_words - stopwords
            answer_content = answer_words - stopwords
            
            if answer_content:
                overlap = len(answer_content & chunk_content) / len(answer_content)
                scores['groundedness'] = round(overlap, 3)
            else:
                scores['groundedness'] = 0.0
        else:
            scores['groundedness'] = 0.5  # No source to check against
        
        # 3. Question-answer relevance: do they share topic terms?
        q_terms = set(question.lower().split()) - {'what','how','why','when','where','who','which','is','are','do','does','the','a','an','can','will'}
        a_terms = set(answer.lower().split()) - {'the','a','an','is','are','it','this','that','to','of','and','in','for'}
        if q_terms:
            relevance = len(q_terms & a_terms) / len(q_terms)
            scores['qa_relevance'] = round(relevance, 3)
        else:
            scores['qa_relevance'] = 0.0
        
        # 4. Format quality (not just regex match, but structural checks)
        scores['ends_with_question_mark'] = question.strip().endswith('?')
        scores['no_generation_artifacts'] = not any(
            artifact in answer.lower() for artifact in 
            ['as an ai', 'i cannot', 'i don\'t know', 'error:', 'note:', 'disclaimer:']
        )
        
        # 5. Overall pass/fail
        scores['overall_score'] = (
            (0.3 * scores.get('groundedness', 0)) +
            (0.3 * scores.get('qa_relevance', 0)) +
            (0.2 * (1.0 if scores['q_length_ok'] else 0.0)) +
            (0.2 * (1.0 if scores['a_length_ok'] else 0.0))
        )
        scores['passed'] = scores['overall_score'] >= self.config.min_qa_confidence
        
        return scores
    
    def filter_qa_batch(self, qa_pairs: List[Dict], source_chunks: Dict[str, str] = None) -> Tuple[List[Dict], Dict]:
        """Filter a batch of QA pairs, returning passed pairs and stats."""
        source_chunks = source_chunks or {}
        passed = []
        stats = {'total': len(qa_pairs), 'passed': 0, 'failed': 0, 'avg_groundedness': 0.0}
        groundedness_scores = []
        
        for qa in qa_pairs:
            question = qa.get('question', qa.get('instruction', ''))
            answer = qa.get('answer', qa.get('output', ''))
            chunk = source_chunks.get(qa.get('source', ''), qa.get('chunk_text', ''))
            
            scores = self.evaluate_qa_pair(question, answer, chunk)
            
            if scores.get('groundedness'):
                groundedness_scores.append(scores['groundedness'])
            
            if scores['passed']:
                qa['qa_quality_score'] = scores['overall_score']
                qa['groundedness_score'] = scores.get('groundedness', 0)
                passed.append(qa)
                stats['passed'] += 1
            else:
                stats['failed'] += 1
        
        stats['avg_groundedness'] = round(np.mean(groundedness_scores), 3) if groundedness_scores else 0.0
        stats['pass_rate'] = round(stats['passed'] / max(stats['total'], 1), 3)
        
        self.logger.info(f"🔍 QA Quality Gate: {stats['passed']}/{stats['total']} passed "
                        f"(groundedness={stats['avg_groundedness']:.2f}, pass_rate={stats['pass_rate']:.1%})")
        
        return passed, stats

qa_gate = QAQualityGate(config, logger)
logger.info("✅ QA quality gate initialized")

## Dataset Governance & Metadata

Generate a HuggingFace-compatible dataset card with:
- License and intended use declaration
- Language coverage and bias considerations
- Pipeline reproducibility metadata

In [ ]:
# Dataset Governance Card Generator
def generate_dataset_card(config: KaggleConfig, data_stats: Dict, output_dir: Path) -> str:
    """Generate a HuggingFace-compatible dataset card (YAML front matter + markdown)."""
    
    card = f"""---
license: cc-by-nc-4.0
language:
  - en
  - lg
tags:
  - tax
  - customer-service
  - uganda
  - ura
  - faq
  - multilingual
size_categories:
  - 1K<n<10K
task_categories:
  - question-answering
  - text-generation
  - translation
dataset_info:
  version: {config.dataset_version}
  random_seed: {config.random_seed}
---

# URA Tax Assistant Training Dataset

## Dataset Description

Training data for the Uganda Revenue Authority (URA) customer-service AI assistant.
Covers tax policy FAQs, PDF document extracts, synthetic QA pairs, and English-Luganda translations.

### Dataset Summary

| Metric | Value |
|--------|-------|
| Total samples | {data_stats.get('total_samples', 'N/A')} |
| FAQ pairs | {data_stats.get('faq_count', 'N/A')} |
| PDF chunks | {data_stats.get('pdf_chunks', 'N/A')} |
| Synthetic QA | {data_stats.get('teacher_qa_count', 'N/A')} |
| Translation pairs | {data_stats.get('translation_count', 'N/A')} |
| Languages | English, Luganda |

### Languages
- **English (en)**: Primary language for tax documentation
- **Luganda (lg)**: Translation pairs for multilingual support

### Data Sources
- URA official FAQ documents (CSV)
- URA policy PDFs (tax guides, regulations)
- Teacher-model generated QA (Qwen2.5-7B-Instruct)
- English-Luganda translation corpus

## Bias, Risks, and Limitations

- Data reflects URA policies as of collection date; tax laws change
- Synthetic QA may contain hallucinations despite quality gates
- Luganda translations are not professionally verified
- PII has been redacted but manual review is recommended

## Pipeline Reproducibility

- **Random seed**: {config.random_seed}
- **Pipeline version**: {config.dataset_version}
- **Dedup method**: Exact hash + MinHash semantic (threshold={config.minhash_threshold})
- **Split strategy**: Stratified by source (no leakage)
- **PII redaction**: {'Enabled' if config.redact_pii else 'Disabled'}
"""
    
    card_path = output_dir / "README.md"
    with open(card_path, 'w', encoding='utf-8') as f:
        f.write(card)
    
    logger.info(f"📋 Dataset card generated: {card_path}")
    return card

logger.info("✅ Governance card generator ready")

## Main Execution Pipeline

Orchestrates the full data pipeline with checkpoint/resume support and progress tracking. If a stage was previously completed and checkpointed, it will be loaded from disk instead of re-executed.

In [ ]:
# Main Execution Pipeline (with checkpointing and validation)
def enhanced_main_pipeline():
    """Enhanced main pipeline with checkpointing, validation, and progress tracking."""
    logger.info("=" * 70)
    logger.info("URA TAX ASSISTANT - ENHANCED DATA PIPELINE")
    logger.info("=" * 70)

    try:
        # Step 1: Load data (with checkpoint)
        with tracker.stage("load_data"):
            cached = checkpoint.load("loaded_data")
            if cached:
                faq_df, pdf_chunks, teacher_qa_df, translations_df = cached
                logger.info("Loaded data from checkpoint")
            else:
                faq_df, pdf_chunks, teacher_qa_df, translations_df = data_loader.load_all_data()
                checkpoint.save("loaded_data", (faq_df, pdf_chunks, teacher_qa_df, translations_df))

            tracker.metric("faq_count", len(faq_df))
            tracker.metric("pdf_chunks", len(pdf_chunks))
            tracker.metric("teacher_qa_count", len(teacher_qa_df))
            tracker.metric("translations_count", len(translations_df))

        # Step 1b: Validate loaded data
        with tracker.stage("validate_data"):
            if PANDERA_AVAILABLE and faq_schema is not None:
                faq_df = validate_dataframe(faq_df, faq_schema, "FAQ")
                if not translations_df.empty:
                    translations_df = validate_dataframe(translations_df, translation_schema, "Translations")
            tracker.metric("faq_valid", len(faq_df))
            tracker.metric("translations_valid", len(translations_df))

        
        # Initialize governance tools
        provenance = DataProvenanceVerifier(config)
        qa_gate = QAQualityGate(min_answer_words=5, max_artifact_ratio=0.1)
        
        # Step 1c: Verify data provenance
        with tracker.stage("verify_provenance"):
            manifest = provenance.generate_manifest(
                config.paths.get("ura_faqs", config.output_dir),
                config.kaggle_dataset_name,
                len(faq_df), len(pdf_chunks), len(translations_df)
            )
            tracker.metric("files_hashed", len(manifest.get("file_checksums", {})))
            tracker.metric("source_trusted", manifest["dataset_source"]["trusted"])

        # Step 2: Generate enhanced teacher QA from PDF chunks
        with tracker.stage("teacher_qa_generation"):
            cached_qa = checkpoint.load("teacher_qa_generated")
            if cached_qa is not None:
                teacher_qa_generated = cached_qa
                logger.info("Loaded teacher QA from checkpoint")
            else:
                teacher_qa_generated = []
                if pdf_chunks:
                    teacher_qa_generated = generate_enhanced_teacher_qa(pdf_chunks, config)
                checkpoint.save("teacher_qa_generated", teacher_qa_generated)

            # Combine existing and generated teacher QA
            all_teacher_qa = []
            if not teacher_qa_df.empty:
                all_teacher_qa.extend(teacher_qa_df.to_dict("records"))
            all_teacher_qa.extend(teacher_qa_generated)
            tracker.metric("total_teacher_qa", len(all_teacher_qa))

            # Apply QA quality gates to generated QA
            if all_teacher_qa:
                all_teacher_qa, qa_stats = qa_gate.filter_qa_batch(all_teacher_qa)
                tracker.metric("qa_gate_pass_rate", qa_stats['pass_rate'])
                tracker.metric("qa_avg_groundedness", qa_stats['avg_groundedness'])

        # Step 3: Augment data
        with tracker.stage("augmentation"):
            augmented_data = augmentor.augment(faq_df, all_teacher_qa, translations_df)
            if not augmented_data:
                logger.error("No augmented data generated!")
                return
            tracker.metric("augmented_samples", len(augmented_data))

        # Step 4: Export datasets
        with tracker.stage("export"):
            output_file = config.output_dir / "enhanced_training_data.jsonl"
            exporter.export_jsonl(augmented_data, output_file)

            split_dir = config.output_dir / "enhanced_splits"
            exporter.export_train_val_split(augmented_data, split_dir)

            stats_file = config.output_dir / "enhanced_dataset_statistics.json"
            stats = exporter.export_statistics(augmented_data, stats_file)

            # Export translation-specific dataset
            translation_data = [d for d in augmented_data if d.get("data_type") == DataType.TRANSLATION.value]
            if translation_data:
                exporter.export_jsonl(translation_data, config.output_dir / "translation_dataset.jsonl")

            # Export teacher QA dataset
            teacher_qa_data = [d for d in augmented_data if d.get("data_type") == DataType.TEACHER_QA.value]
            if teacher_qa_data:
                exporter.export_jsonl(teacher_qa_data, config.output_dir / "teacher_qa_dataset.jsonl")

        # Pipeline summary
        # Step 5: Generate governance metadata
        with tracker.stage("governance"):
            stats = {
                "total_samples": len(augmented_data),
                "faq_count": len(faq_df),
                "pdf_chunks": len(pdf_chunks),
                "teacher_qa_count": len(all_teacher_qa) if 'all_teacher_qa' in dir() else 0,
                "translation_count": len(translations_df),
            }
            generate_dataset_card(config, stats, config.output_dir)

        summary = tracker.summary()
        summary_path = config.output_dir / "pipeline_summary.json"
        with open(summary_path, "w") as f:
            json.dump(summary, f, indent=2, default=str)

        logger.info("\n" + "=" * 70)
        logger.info("PIPELINE COMPLETE")
        logger.info("=" * 70)
        logger.info(f"Total time: {summary['total_elapsed_s']:.1f}s")
        logger.info(f"Total samples: {len(augmented_data)}")
        logger.info(f"Output: {output_file}")
        logger.info(f"Summary: {summary_path}")

        if torch.cuda.is_available():
            for i in range(torch.cuda.device_count()):
                mem_alloc = torch.cuda.memory_allocated(i) / 1e9
                logger.info(f"GPU {i}: {mem_alloc:.2f}GB allocated")

    except Exception as e:
        logger.error(f"Pipeline failed: {e}", exc_info=True)
        raise

# Run the enhanced pipeline
if __name__ == "__main__":
    enhanced_main_pipeline()


In [ ]:
# Cell 11: Quick Test Function (Optional)
def quick_test():
    """Quick test to verify the pipeline works."""
    logger.info("🧪 Running quick test...")
    
    # Test data loading
    faq_df, pdf_chunks, teacher_qa_df, translations_df = data_loader.load_all_data()
    
    logger.info(f"📊 Quick Test Results:")
    logger.info(f"  • FAQ pairs: {len(faq_df)}")
    logger.info(f"  • PDF chunks: {len(pdf_chunks)}")
    logger.info(f"  • Teacher QA: {len(teacher_qa_df)}")
    logger.info(f"  • Translations: {len(translations_df)}")
    
    # Test augmentor on small sample
    if not faq_df.empty:
        sample_faq = faq_df.head(3)
        sample_augmented = augmentor.augment(sample_faq, [], translations_df.head(2) if not translations_df.empty else pd.DataFrame())
        logger.info(f"  • Augmented samples from 3 FAQ pairs: {len(sample_augmented)}")
    
    logger.info("✅ Quick test completed!")

# Uncomment to run quick test
# quick_test()

## Exploratory Data Analysis (EDA)

IEEE publication-quality visualizations covering:
- Data source composition and distribution
- Text length distributions across sources
- Word clouds and tax-term frequency analysis
- Interactive Plotly visualizations
- QA quality assessment metrics
- Comprehensive markdown EDA report

In [ ]:
# Cell 12: EDA Visualizations Setup
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd
from collections import Counter
import numpy as np

# Set plotting style for IEEE publication
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({
    'figure.figsize': (10, 6),
    'figure.dpi': 300,
    'font.size': 10,
    'axes.titlesize': 12,
    'axes.labelsize': 10,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'legend.fontsize': 9,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'savefig.pad_inches': 0.1
})

# Color palette for IEEE (perceptually uniform)
IEEE_COLORS = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf']

def save_ieee_figure(fig, filename, dpi=300):
    """Save figure with IEEE publication standards."""
    fig.savefig(filename, dpi=dpi, bbox_inches='tight', pad_inches=0.05)
    plt.close(fig)
    logger.info(f"📊 Saved IEEE figure: {filename}")

# Initialize EDA results directory
eda_dir = config.output_dir / "eda_visualizations"
eda_dir.mkdir(exist_ok=True)

In [ ]:
# Cell 13: Data Composition Analysis
def visualize_data_composition(faq_df, pdf_chunks, teacher_qa_df, translations_df):
    """Create comprehensive data composition visualizations."""
    
    logger.info("📊 Generating data composition visualizations...")
    
    # Prepare data
    data_sources = {
        'FAQ': len(faq_df) if not faq_df.empty else 0,
        'PDF Chunks': len(pdf_chunks) if pdf_chunks else 0,
        'Teacher QA': len(teacher_qa_df) if not teacher_qa_df.empty else 0,
        'Translations': len(translations_df) if not translations_df.empty else 0
    }
    
    # 1. Pie chart for data source distribution
    fig1, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    
    # Pie chart
    valid_sources = {k: v for k, v in data_sources.items() if v > 0}
    colors = IEEE_COLORS[:len(valid_sources)]
    wedges, texts, autotexts = ax1.pie(
        valid_sources.values(),
        labels=valid_sources.keys(),
        autopct='%1.1f%%',
        colors=colors,
        startangle=90,
        wedgeprops=dict(width=0.3, edgecolor='w')
    )
    
    # Improve text visibility
    for autotext in autotexts:
        autotext.set_color('white')
        autotext.set_fontweight('bold')
    
    ax1.set_title('Data Source Distribution', fontweight='bold')
    
    # 2. Bar chart with counts
    sources = list(valid_sources.keys())
    counts = list(valid_sources.values())
    
    bars = ax2.bar(sources, counts, color=colors, edgecolor='black', linewidth=0.5)
    ax2.set_title('Data Source Counts', fontweight='bold')
    ax2.set_ylabel('Number of Samples')
    ax2.set_xlabel('Data Source')
    ax2.grid(True, alpha=0.3, linestyle='--')
    
    # Add count labels on bars
    for bar, count in zip(bars, counts):
        height = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2., height + max(counts)*0.01,
                f'{count:,}', ha='center', va='bottom', fontsize=9)
    
    plt.tight_layout()
    save_ieee_figure(fig1, eda_dir / "data_composition.png")
    plt.show()
    
    # 3. Text statistics table
    logger.info("\n📈 Text Statistics:")
    logger.info("-" * 50)
    
    stats_data = []
    
    # FAQ statistics
    if not faq_df.empty:
        faq_q_lengths = faq_df['question'].str.split().str.len()
        faq_a_lengths = faq_df['answer'].str.split().str.len()
        stats_data.append({
            'Source': 'FAQ',
            'Samples': len(faq_df),
            'Avg Q Length': f"{faq_q_lengths.mean():.1f}",
            'Avg A Length': f"{faq_a_lengths.mean():.1f}",
            'Q Length Std': f"{faq_q_lengths.std():.1f}",
            'A Length Std': f"{faq_a_lengths.std():.1f}"
        })
    
    # PDF statistics
    if pdf_chunks:
        pdf_df = pd.DataFrame(pdf_chunks)
        if 'word_count' in pdf_df.columns:
            pdf_stats = pdf_df['word_count'].describe()
            stats_data.append({
                'Source': 'PDF Chunks',
                'Samples': len(pdf_chunks),
                'Avg Words': f"{pdf_stats['mean']:.1f}",
                'Min Words': f"{pdf_stats['min']:.0f}",
                'Max Words': f"{pdf_stats['max']:.0f}",
                'Std Words': f"{pdf_stats['std']:.1f}"
            })
    
    # Teacher QA statistics
    if not teacher_qa_df.empty:
        tq_q_lengths = teacher_qa_df['question'].str.split().str.len()
        tq_a_lengths = teacher_qa_df['answer'].str.split().str.len()
        stats_data.append({
            'Source': 'Teacher QA',
            'Samples': len(teacher_qa_df),
            'Avg Q Length': f"{tq_q_lengths.mean():.1f}",
            'Avg A Length': f"{tq_a_lengths.mean():.1f}",
            'Q Length Std': f"{tq_q_lengths.std():.1f}",
            'A Length Std': f"{tq_a_lengths.std():.1f}"
        })
    
    # Translation statistics
    if not translations_df.empty:
        trans_en = translations_df['english'].str.split().str.len()
        trans_lg = translations_df['luganda'].str.split().str.len()
        stats_data.append({
            'Source': 'Translations',
            'Samples': len(translations_df),
            'Avg EN Length': f"{trans_en.mean():.1f}",
            'Avg LG Length': f"{trans_lg.mean():.1f}",
            'EN:LG Ratio': f"{(trans_lg/trans_en).mean():.2f}",
            'Pairs with LG > EN': f"{(trans_lg > trans_en).sum()}"
        })
    
    # Create and display table
    stats_df = pd.DataFrame(stats_data)
    logger.info(f"\n{stats_df.to_string(index=False)}")
    
    # Save statistics to CSV
    stats_df.to_csv(eda_dir / "text_statistics.csv", index=False)
    logger.info(f"\n💾 Saved text statistics to: {eda_dir}/text_statistics.csv")
    
    return stats_df

In [ ]:
# Cell 14: Text Length Distribution Analysis
def visualize_text_distributions(faq_df, pdf_chunks, teacher_qa_df, translations_df):
    """Visualize text length distributions across data sources."""
    
    logger.info("📏 Analyzing text length distributions...")
    
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    axes = axes.flatten()
    
    plot_index = 0
    
    # 1. FAQ Length Distribution
    if not faq_df.empty:
        faq_q_lengths = faq_df['question'].str.split().str.len()
        faq_a_lengths = faq_df['answer'].str.split().str.len()
        
        ax = axes[plot_index]
        ax.hist([faq_q_lengths, faq_a_lengths], 
                bins=30, alpha=0.7, 
                label=['Questions', 'Answers'],
                color=[IEEE_COLORS[0], IEEE_COLORS[1]],
                edgecolor='black', linewidth=0.5)
        
        ax.set_title('FAQ: Question vs Answer Length Distribution', fontweight='bold')
        ax.set_xlabel('Number of Words')
        ax.set_ylabel('Frequency')
        ax.legend()
        ax.grid(True, alpha=0.3, linestyle='--')
        plot_index += 1
    
    # 2. PDF Chunk Length Distribution
    if pdf_chunks:
        pdf_df = pd.DataFrame(pdf_chunks)
        if 'word_count' in pdf_df.columns:
            ax = axes[plot_index]
            ax.hist(pdf_df['word_count'], bins=30, alpha=0.7,
                   color=IEEE_COLORS[2], edgecolor='black', linewidth=0.5)
            
            # Add vertical line for mean
            mean_val = pdf_df['word_count'].mean()
            ax.axvline(mean_val, color='red', linestyle='--', linewidth=1.5,
                      label=f'Mean: {mean_val:.1f}')
            
            ax.set_title('PDF Chunks: Word Count Distribution', fontweight='bold')
            ax.set_xlabel('Number of Words')
            ax.set_ylabel('Frequency')
            ax.legend()
            ax.grid(True, alpha=0.3, linestyle='--')
            plot_index += 1
    
    # 3. Translation Length Comparison
    if not translations_df.empty and plot_index < 4:
        trans_en = translations_df['english'].str.split().str.len()
        trans_lg = translations_df['luganda'].str.split().str.len()
        
        ax = axes[plot_index]
        
        # Scatter plot
        ax.scatter(trans_en, trans_lg, alpha=0.6, color=IEEE_COLORS[3],
                  edgecolor='black', linewidth=0.3)
        
        # Add diagonal line (y = x)
        max_val = max(trans_en.max(), trans_lg.max())
        ax.plot([0, max_val], [0, max_val], 'r--', alpha=0.5, linewidth=1.5,
               label='Equal Length (y = x)')
        
        # Add regression line
        if len(trans_en) > 2:
            try:
                z = np.polyfit(trans_en, trans_lg, 1)
                p = np.poly1d(z)
                sorted_en = np.sort(trans_en)
                ax.plot(sorted_en, p(sorted_en), "g--", alpha=0.7, linewidth=1.5,
                       label=f'Fit: y = {z[0]:.2f}x + {z[1]:.2f}')
            except (np.linalg.LinAlgError, ValueError):
                logger.warning("Could not compute regression line for translations")
        
        ax.set_title('Translation: EN vs LG Word Count', fontweight='bold')
        ax.set_xlabel('English Word Count')
        ax.set_ylabel('Luganda Word Count')
        ax.legend()
        ax.grid(True, alpha=0.3, linestyle='--')
        plot_index += 1
    
    # 4. Teacher QA Confidence Distribution (if available)
    if not teacher_qa_df.empty and 'confidence' in teacher_qa_df.columns and plot_index < 4:
        confidences = teacher_qa_df['confidence']
        
        ax = axes[plot_index]
        ax.hist(confidences, bins=20, alpha=0.7,
               color=IEEE_COLORS[4], edgecolor='black', linewidth=0.5)
        
        # Add vertical line for mean
        mean_conf = confidences.mean()
        ax.axvline(mean_conf, color='red', linestyle='--', linewidth=1.5,
                  label=f'Mean: {mean_conf:.2f}')
        
        ax.set_title('Teacher QA: Confidence Score Distribution', fontweight='bold')
        ax.set_xlabel('Confidence Score')
        ax.set_ylabel('Frequency')
        ax.legend()
        ax.grid(True, alpha=0.3, linestyle='--')
    
    # Remove empty subplots
    for i in range(plot_index, 4):
        fig.delaxes(axes[i])
    
    plt.tight_layout()
    save_ieee_figure(fig, eda_dir / "text_distributions.png")
    plt.show()
    plt.close('all')
    
    # Print summary statistics
    logger.info("\n📊 Summary Statistics:")
    logger.info("-" * 50)
    
    if not faq_df.empty:
        logger.info(f"FAQ Questions: Mean={faq_q_lengths.mean():.1f} words, "
                   f"Std={faq_q_lengths.std():.1f}, Range={faq_q_lengths.min()}-{faq_q_lengths.max()}")
        logger.info(f"FAQ Answers: Mean={faq_a_lengths.mean():.1f} words, "
                   f"Std={faq_a_lengths.std():.1f}, Range={faq_a_lengths.min()}-{faq_a_lengths.max()}")
    
    if pdf_chunks and 'word_count' in pdf_df.columns:
        logger.info(f"PDF Chunks: Mean={pdf_df['word_count'].mean():.1f} words, "
                   f"Std={pdf_df['word_count'].std():.1f}, Range={pdf_df['word_count'].min()}-{pdf_df['word_count'].max()}")
    
    if not translations_df.empty:
        logger.info(f"Translations EN:LG Ratio: Mean={(trans_lg/trans_en).mean():.2f}, "
                   f"Std={(trans_lg/trans_en).std():.2f}")

In [ ]:
# Cell 15: Word Cloud and Term Frequency Analysis
def visualize_term_frequencies(faq_df, pdf_chunks, translations_df):
    """Generate word clouds and term frequency visualizations."""
    
    logger.info("🔤 Analyzing term frequencies and generating word clouds...")
    
    fig = plt.figure(figsize=(15, 12))
    
    # 1. FAQ Word Cloud
    if not faq_df.empty:
        ax1 = plt.subplot(2, 2, 1)
        all_faq_text = ' '.join(faq_df['question'].tolist() + faq_df['answer'].tolist())
        
        wordcloud = WordCloud(
            width=800, height=400,
            background_color='white',
            colormap='viridis',
            max_words=100,
            contour_width=1,
            contour_color='steelblue'
        ).generate(all_faq_text)
        
        ax1.imshow(wordcloud, interpolation='bilinear')
        ax1.set_title('FAQ: Most Frequent Terms', fontweight='bold')
        ax1.axis('off')
    
    # 2. PDF Content Word Cloud
    if pdf_chunks:
        ax2 = plt.subplot(2, 2, 2)
        pdf_texts = [chunk['text'] for chunk in pdf_chunks[:50]]  # Sample first 50
        all_pdf_text = ' '.join(pdf_texts)
        
        wordcloud = WordCloud(
            width=800, height=400,
            background_color='white',
            colormap='plasma',
            max_words=100,
            contour_width=1,
            contour_color='darkred'
        ).generate(all_pdf_text)
        
        ax2.imshow(wordcloud, interpolation='bilinear')
        ax2.set_title('PDF Content: Most Frequent Terms', fontweight='bold')
        ax2.axis('off')
    
    # 3. Tax Terminology Frequency
    ax3 = plt.subplot(2, 2, 3)
    
    tax_terms = {
        'tax': 0, 'vat': 0, 'tin': 0, 'ura': 0, 'payment': 0,
        'return': 0, 'business': 0, 'income': 0, 'registration': 0,
        'compliance': 0, 'deadline': 0, 'penalty': 0, 'assessment': 0
    }
    
    # Count tax terms in FAQ
    if not faq_df.empty:
        combined_text = ' '.join(faq_df['question'].str.lower().tolist() + 
                                faq_df['answer'].str.lower().tolist())
        
        for term in tax_terms.keys():
            tax_terms[term] = combined_text.count(term)
    
    # Create bar chart
    terms = list(tax_terms.keys())
    counts = list(tax_terms.values())
    
    bars = ax3.barh(terms, counts, color=IEEE_COLORS[:len(terms)], edgecolor='black', linewidth=0.5)
    ax3.set_title('Tax Terminology Frequency in FAQ', fontweight='bold')
    ax3.set_xlabel('Frequency Count')
    ax3.invert_yaxis()  # Highest frequency on top
    ax3.grid(True, alpha=0.3, linestyle='--', axis='x')
    
    # Add count labels
    for bar, count in zip(bars, counts):
        width = bar.get_width()
        ax3.text(width + max(counts)*0.01, bar.get_y() + bar.get_height()/2,
                f'{count}', ha='left', va='center', fontsize=8)
    
    # 4. Translation Language Analysis
    if not translations_df.empty:
        ax4 = plt.subplot(2, 2, 4)
        
        # Calculate average word lengths
        en_word_counts = translations_df['english'].str.split().str.len()
        lg_word_counts = translations_df['luganda'].str.split().str.len()
        
        # Box plot comparison
        bp = ax4.boxplot([en_word_counts, lg_word_counts], 
                        labels=['English', 'Luganda'],
                        patch_artist=True,
                        widths=0.6)
        
        # Customize box colors
        colors = [IEEE_COLORS[0], IEEE_COLORS[1]]
        for patch, color in zip(bp['boxes'], colors):
            patch.set_facecolor(color)
            patch.set_alpha(0.7)
        
        # Customize median lines
        for median in bp['medians']:
            median.set_color('black')
            median.set_linewidth(1.5)
        
        ax4.set_title('Translation: Word Count Distribution by Language', fontweight='bold')
        ax4.set_ylabel('Number of Words')
        ax4.grid(True, alpha=0.3, linestyle='--', axis='y')
        
        # Add mean markers
        en_mean = en_word_counts.mean()
        lg_mean = lg_word_counts.mean()
        ax4.scatter([1], [en_mean], color='red', s=100, zorder=3, label='Mean')
        ax4.scatter([2], [lg_mean], color='red', s=100, zorder=3)
        
        # Add mean values as text
        ax4.text(1, en_mean + max(en_word_counts.max(), lg_word_counts.max())*0.05,
                f'Mean: {en_mean:.1f}', ha='center', fontsize=8)
        ax4.text(2, lg_mean + max(en_word_counts.max(), lg_word_counts.max())*0.05,
                f'Mean: {lg_mean:.1f}', ha='center', fontsize=8)
    
    plt.tight_layout()
    save_ieee_figure(fig, eda_dir / "term_frequencies.png")
    plt.show()
    
    # Save term frequencies to CSV
    if tax_terms:
        term_df = pd.DataFrame(list(tax_terms.items()), columns=['Term', 'Frequency'])
        term_df = term_df.sort_values('Frequency', ascending=False)
        term_df.to_csv(eda_dir / "tax_term_frequencies.csv", index=False)
        logger.info(f"💾 Saved tax term frequencies to: {eda_dir}/tax_term_frequencies.csv")

In [ ]:
# Cell 16: Interactive Visualizations with Plotly
def create_interactive_visualizations(faq_df, pdf_chunks, translations_df):
    """Create interactive visualizations using Plotly."""
    
    logger.info("📈 Generating interactive visualizations...")
    
    # 1. Interactive Scatter Plot for Translations
    if not translations_df.empty:
        translations_df['en_word_count'] = translations_df['english'].str.split().str.len()
        translations_df['lg_word_count'] = translations_df['luganda'].str.split().str.len()
        translations_df['length_ratio'] = translations_df['lg_word_count'] / translations_df['en_word_count']
        
        fig1 = px.scatter(translations_df, 
                         x='en_word_count', 
                         y='lg_word_count',
                         color='length_ratio',
                         size='en_word_count',
                         hover_data=['english', 'luganda', 'source'],
                         title='Translation Length Analysis: English vs Luganda',
                         labels={'en_word_count': 'English Word Count',
                                'lg_word_count': 'Luganda Word Count',
                                'length_ratio': 'LG:EN Ratio'},
                         color_continuous_scale='Viridis')
        
        # Add diagonal line
        max_val = max(translations_df['en_word_count'].max(), 
                     translations_df['lg_word_count'].max())
        fig1.add_trace(go.Scatter(x=[0, max_val], y=[0, max_val],
                                 mode='lines',
                                 line=dict(color='red', dash='dash'),
                                 name='Equal Length'))
        
        fig1.update_layout(template='plotly_white',
                          font=dict(size=10),
                          showlegend=True)
        
        # Save interactive HTML
        fig1.write_html(str(eda_dir / "translation_length_analysis.html"))
        logger.info(f"💾 Saved interactive plot to: {eda_dir}/translation_length_analysis.html")
        
        # Show in notebook
        fig1.show()
    
    # 2. FAQ Category Distribution (if categories exist)
    if not faq_df.empty and 'category' in faq_df.columns:
        category_counts = faq_df['category'].value_counts().reset_index()
        category_counts.columns = ['Category', 'Count']
        
        fig2 = px.bar(category_counts.head(10),  # Top 10 categories
                     x='Category', 
                     y='Count',
                     color='Count',
                     color_continuous_scale='Blues',
                     title='FAQ Category Distribution (Top 10)',
                     labels={'Count': 'Number of Q/A Pairs'})
        
        fig2.update_layout(template='plotly_white',
                          font=dict(size=10),
                          xaxis_tickangle=-45)
        
        fig2.write_html(str(eda_dir / "faq_category_distribution.html"))
        logger.info(f"💾 Saved interactive plot to: {eda_dir}/faq_category_distribution.html")
        
        fig2.show()
    
    # 3. PDF Chunk Word Count Distribution
    if pdf_chunks:
        pdf_word_counts = [chunk.get('word_count', len(chunk['text'].split())) 
                          for chunk in pdf_chunks]
        
        fig3 = go.Figure()
        
        # Histogram
        fig3.add_trace(go.Histogram(x=pdf_word_counts,
                                   nbinsx=50,
                                   name='PDF Chunks',
                                   marker_color=IEEE_COLORS[0],
                                   opacity=0.7))
        
        # Add box plot above
        fig3.add_trace(go.Box(x=pdf_word_counts,
                             name='Distribution',
                             marker_color=IEEE_COLORS[1],
                             boxpoints=False))
        
        fig3.update_layout(title='PDF Chunk Word Count Distribution',
                          xaxis_title='Word Count',
                          yaxis_title='Frequency',
                          template='plotly_white',
                          font=dict(size=10),
                          showlegend=True,
                          hovermode='x unified')
        
        fig3.write_html(str(eda_dir / "pdf_chunk_distribution.html"))
        logger.info(f"💾 Saved interactive plot to: {eda_dir}/pdf_chunk_distribution.html")
        
        fig3.show()
    
    # 4. Correlation Heatmap for Text Features
    if not faq_df.empty and not translations_df.empty:
        # Prepare data for correlation analysis
        corr_data = []
        
        # FAQ features
        faq_df['q_len'] = faq_df['question'].str.split().str.len()
        faq_df['a_len'] = faq_df['answer'].str.split().str.len()
        faq_df['total_len'] = faq_df['q_len'] + faq_df['a_len']
        faq_df['q_a_ratio'] = faq_df['q_len'] / faq_df['a_len']
        
        # Translation features
        if not translations_df.empty:
            translations_df['en_len'] = translations_df['english'].str.split().str.len()
            translations_df['lg_len'] = translations_df['luganda'].str.split().str.len()
            translations_df['trans_ratio'] = translations_df['lg_len'] / translations_df['en_len']
        
        # Create correlation matrix
        import warnings
        warnings.filterwarnings('ignore')
        
        # Simplified correlation for demo
        logger.info("\n📊 Text Feature Correlations:")
        logger.info("-" * 50)
        
        if not faq_df.empty:
            faq_corr = faq_df[['q_len', 'a_len', 'total_len']].corr()
            logger.info(f"\nFAQ Feature Correlations:\n{faq_corr}")
            
            # Create heatmap
            fig4 = px.imshow(faq_corr,
                            text_auto=True,
                            color_continuous_scale='RdBu',
                            title='FAQ Text Feature Correlation Matrix',
                            labels=dict(color="Correlation"))
            
            fig4.update_layout(template='plotly_white',
                              font=dict(size=10))
            
            fig4.write_html(str(eda_dir / "faq_feature_correlation.html"))
            logger.info(f"💾 Saved interactive plot to: {eda_dir}/faq_feature_correlation.html")
            
            fig4.show()

In [ ]:
# Cell 17: Quality Assessment Metrics
def visualize_quality_metrics(faq_df, teacher_qa_df):
    """Visualize quality assessment metrics for the dataset."""
    
    logger.info("🎯 Analyzing dataset quality metrics...")
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    # 1. Question Quality Metrics
    if not faq_df.empty:
        faq_q_lengths = faq_df['question'].str.split().str.len()
        faq_a_lengths = faq_df['answer'].str.split().str.len()
        
        # Question word count distribution
        axes[0].hist(faq_q_lengths, bins=30, alpha=0.7, color=IEEE_COLORS[0],
                    edgecolor='black', linewidth=0.5)
        axes[0].axvline(faq_q_lengths.mean(), color='red', linestyle='--',
                       linewidth=1.5, label=f'Mean: {faq_q_lengths.mean():.1f}')
        axes[0].set_title('FAQ Question Length Distribution', fontweight='bold')
        axes[0].set_xlabel('Word Count')
        axes[0].set_ylabel('Frequency')
        axes[0].legend()
        axes[0].grid(True, alpha=0.3, linestyle='--')
    
    # 2. Answer Quality Metrics
    if not faq_df.empty:
        axes[1].hist(faq_a_lengths, bins=30, alpha=0.7, color=IEEE_COLORS[1],
                    edgecolor='black', linewidth=0.5)
        axes[1].axvline(faq_a_lengths.mean(), color='red', linestyle='--',
                       linewidth=1.5, label=f'Mean: {faq_a_lengths.mean():.1f}')
        axes[1].set_title('FAQ Answer Length Distribution', fontweight='bold')
        axes[1].set_xlabel('Word Count')
        axes[1].set_ylabel('Frequency')
        axes[1].legend()
        axes[1].grid(True, alpha=0.3, linestyle='--')
    
    # 3. Q/A Ratio Analysis
    if not faq_df.empty:
        qa_ratios = faq_a_lengths / faq_q_lengths
        
        axes[2].hist(qa_ratios, bins=30, alpha=0.7, color=IEEE_COLORS[2],
                    edgecolor='black', linewidth=0.5)
        axes[2].axvline(qa_ratios.mean(), color='red', linestyle='--',
                       linewidth=1.5, label=f'Mean Ratio: {qa_ratios.mean():.2f}')
        axes[2].axvline(1.0, color='green', linestyle=':', linewidth=1.5,
                       label='Equal Length (Ratio = 1)')
        axes[2].set_title('FAQ Answer-to-Question Length Ratio', fontweight='bold')
        axes[2].set_xlabel('Answer Length / Question Length')
        axes[2].set_ylabel('Frequency')
        axes[2].legend()
        axes[2].grid(True, alpha=0.3, linestyle='--')
    
    plt.tight_layout()
    save_ieee_figure(fig, eda_dir / "quality_metrics.png")
    plt.show()
    
    # Print quality summary
    logger.info("\n📊 QUALITY ASSESSMENT SUMMARY:")
    logger.info("-" * 50)
    
    if not faq_df.empty:
        logger.info(f"1. Question Length Analysis:")
        logger.info(f"   • Average question length: {faq_q_lengths.mean():.1f} words")
        logger.info(f"   • Standard deviation: {faq_q_lengths.std():.1f} words")
        logger.info(f"   • Range: {faq_q_lengths.min()} - {faq_q_lengths.max()} words")
        
        logger.info(f"\n2. Answer Length Analysis:")
        logger.info(f"   • Average answer length: {faq_a_lengths.mean():.1f} words")
        logger.info(f"   • Standard deviation: {faq_a_lengths.std():.1f} words")
        logger.info(f"   • Range: {faq_a_lengths.min()} - {faq_a_lengths.max()} words")
        
        logger.info(f"\n3. Q/A Ratio Analysis:")
        logger.info(f"   • Average A/Q ratio: {qa_ratios.mean():.2f}")
        logger.info(f"   • Median A/Q ratio: {qa_ratios.median():.2f}")
        logger.info(f"   • % with answers longer than questions: {(qa_ratios > 1).mean()*100:.1f}%")
        
        # Quality categories based on A/Q ratio
        excellent = (qa_ratios >= 1.5).sum()
        good = ((qa_ratios >= 1.0) & (qa_ratios < 1.5)).sum()
        fair = ((qa_ratios >= 0.5) & (qa_ratios < 1.0)).sum()
        poor = (qa_ratios < 0.5).sum()
        
        logger.info(f"\n4. Quality Categorization (by A/Q ratio):")
        logger.info(f"   • Excellent (ratio ≥ 1.5): {excellent} samples ({excellent/len(faq_df)*100:.1f}%)")
        logger.info(f"   • Good (1.0 ≤ ratio < 1.5): {good} samples ({good/len(faq_df)*100:.1f}%)")
        logger.info(f"   • Fair (0.5 ≤ ratio < 1.0): {fair} samples ({fair/len(faq_df)*100:.1f}%)")
        logger.info(f"   • Poor (ratio < 0.5): {poor} samples ({poor/len(faq_df)*100:.1f}%)")
    
    # Teacher QA confidence analysis
    if not teacher_qa_df.empty and 'confidence' in teacher_qa_df.columns:
        confidences = teacher_qa_df['confidence']
        
        logger.info(f"\n5. Teacher QA Confidence Analysis:")
        logger.info(f"   • Average confidence: {confidences.mean():.3f}")
        logger.info(f"   • Confidence distribution:")
        logger.info(f"     - ≥ 0.9: {(confidences >= 0.9).sum()} samples")
        logger.info(f"     - 0.7-0.9: {((confidences >= 0.7) & (confidences < 0.9)).sum()} samples")
        logger.info(f"     - 0.5-0.7: {((confidences >= 0.5) & (confidences < 0.7)).sum()} samples")
        logger.info(f"     - < 0.5: {(confidences < 0.5).sum()} samples")

In [ ]:
# Cell 18: Export EDA Report
def generate_eda_report(faq_df, pdf_chunks, teacher_qa_df, translations_df):
    """Generate comprehensive EDA report."""
    
    logger.info("📋 Generating comprehensive EDA report...")
    
    report_file = eda_dir / "eda_report.md"
    
    with open(report_file, 'w', encoding='utf-8') as f:
        f.write("# URA Tax Assistant - Exploratory Data Analysis Report\n\n")
        f.write(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")
        
        f.write("## 1. Dataset Overview\n\n")
        f.write("| Data Source | Samples | Description |\n")
        f.write("|------------|---------|-------------|\n")
        
        if not faq_df.empty:
            f.write(f"| FAQ Pairs | {len(faq_df):,} | Structured question-answer pairs |\n")
        
        if pdf_chunks:
            f.write(f"| PDF Chunks | {len(pdf_chunks):,} | Extracted and chunked PDF content |\n")
        
        if not teacher_qa_df.empty:
            f.write(f"| Teacher QA | {len(teacher_qa_df):,} | Model-generated QA pairs |\n")
        
        if not translations_df.empty:
            f.write(f"| Translations | {len(translations_df):,} | English-Luganda parallel data |\n")
        
        f.write("\n## 2. Data Quality Metrics\n\n")
        
        if not faq_df.empty:
            faq_q_len = faq_df['question'].str.split().str.len()
            faq_a_len = faq_df['answer'].str.split().str.len()
            qa_ratio = faq_a_len / faq_q_len
            
            f.write("### FAQ Data Quality\n")
            f.write(f"- Average question length: {faq_q_len.mean():.1f} words\n")
            f.write(f"- Average answer length: {faq_a_len.mean():.1f} words\n")
            f.write(f"- Average answer-to-question ratio: {qa_ratio.mean():.2f}\n")
            f.write(f"- Questions with sufficient length (>5 words): {(faq_q_len > 5).sum()}/{len(faq_q_len)} ({((faq_q_len > 5).sum()/len(faq_q_len)*100):.1f}%)\n")
            f.write(f"- Answers with sufficient length (>10 words): {(faq_a_len > 10).sum()}/{len(faq_a_len)} ({((faq_a_len > 10).sum()/len(faq_a_len)*100):.1f}%)\n\n")
        
        if not translations_df.empty:
            en_len = translations_df['english'].str.split().str.len()
            lg_len = translations_df['luganda'].str.split().str.len()
            
            f.write("### Translation Data Quality\n")
            f.write(f"- Average English text length: {en_len.mean():.1f} words\n")
            f.write(f"- Average Luganda text length: {lg_len.mean():.1f} words\n")
            f.write(f"- Average length ratio (LG:EN): {(lg_len/en_len).mean():.2f}\n")
            f.write(f"- Complete translation pairs: {len(translations_df)}\n")
            f.write(f"- Valid translation pairs (LG ≠ EN): {(lg_len != en_len).sum()}\n\n")
        
        f.write("## 3. Generated Visualizations\n\n")
        f.write("The following visualizations have been generated:\n\n")
        
        viz_files = [
            ("Data Composition", "data_composition.png"),
            ("Text Distributions", "text_distributions.png"),
            ("Term Frequencies", "term_frequencies.png"),
            ("Quality Metrics", "quality_metrics.png")
        ]
        
        for viz_name, viz_file in viz_files:
            if (eda_dir / viz_file).exists():
                f.write(f"### {viz_name}\n")
                f.write(f"![{viz_name}]({viz_file})\n\n")
        
        f.write("## 4. Interactive Visualizations\n\n")
        f.write("Interactive HTML visualizations are available in the `eda_visualizations` directory:\n\n")
        
        html_files = [
            "translation_length_analysis.html",
            "faq_category_distribution.html",
            "pdf_chunk_distribution.html",
            "faq_feature_correlation.html"
        ]
        
        for html_file in html_files:
            if (eda_dir / html_file).exists():
                f.write(f"- [{html_file}]({html_file})\n")
        
        f.write("\n## 5. Data Statistics Files\n\n")
        f.write("The following CSV files contain detailed statistics:\n\n")
        
        csv_files = [
            ("Text Statistics", "text_statistics.csv"),
            ("Tax Term Frequencies", "tax_term_frequencies.csv")
        ]
        
        for csv_name, csv_file in csv_files:
            if (eda_dir / csv_file).exists():
                f.write(f"- [{csv_name}]({csv_file})\n")
        
        f.write("\n## 6. Recommendations\n\n")
        
        if not faq_df.empty:
            faq_q_len = faq_df['question'].str.split().str.len()
            faq_a_len = faq_df['answer'].str.split().str.len()
            
            short_questions = (faq_q_len < 3).sum()
            short_answers = (faq_a_len < 5).sum()
            
            if short_questions > 0:
                f.write(f"- **Consider enhancing {short_questions} FAQ questions** that are too short (< 3 words)\n")
            
            if short_answers > 0:
                f.write(f"- **Consider enhancing {short_answers} FAQ answers** that are too short (< 5 words)\n")
        
        if pdf_chunks:
            pdf_word_counts = [chunk.get('word_count', len(chunk['text'].split())) 
                             for chunk in pdf_chunks]
            
            short_chunks = sum(1 for wc in pdf_word_counts if wc < config.min_chunk_words)
            if short_chunks > 0:
                f.write(f"- **Review {short_chunks} PDF chunks** that are below minimum word count ({config.min_chunk_words} words)\n")
        
        f.write("\n## 7. Next Steps\n\n")
        f.write("1. Review the quality metrics and address any identified issues\n")
        f.write("2. Use the augmented datasets for model training\n")
        f.write("3. Monitor model performance on different data types\n")
        f.write("4. Iteratively improve data quality based on model evaluation\n")
    
    logger.info(f"📋 EDA report generated: {report_file}")
    
    # Print summary
    logger.info("\n" + "="*60)
    logger.info("EDA VISUALIZATION COMPLETE")
    logger.info("="*60)
    logger.info(f"📁 All visualizations saved to: {eda_dir}")
    logger.info(f"📋 Report: {report_file}")
    logger.info("📊 Interactive visualizations available as HTML files")
    logger.info("="*60)

## Full Pipeline with EDA

Combines the data pipeline with EDA. This is the primary entry point when running the notebook end-to-end.

In [ ]:
# Enhanced Main Pipeline with EDA (with checkpointing and validation)
def enhanced_main_pipeline_with_eda():
    """Enhanced main pipeline with integrated EDA, checkpointing, and validation."""

    logger.info("=" * 70)
    logger.info("URA TAX ASSISTANT - ENHANCED DATA PIPELINE WITH EDA")
    logger.info("=" * 70)

    try:
        # Step 1: Load data
        with tracker.stage("load_data"):
            cached = checkpoint.load("loaded_data")
            if cached:
                faq_df, pdf_chunks, teacher_qa_df, translations_df = cached
                logger.info("Loaded data from checkpoint")
            else:
                faq_df, pdf_chunks, teacher_qa_df, translations_df = data_loader.load_all_data()
                checkpoint.save("loaded_data", (faq_df, pdf_chunks, teacher_qa_df, translations_df))

            tracker.metric("faq_count", len(faq_df))
            tracker.metric("pdf_chunks", len(pdf_chunks))
            tracker.metric("teacher_qa_count", len(teacher_qa_df))
            tracker.metric("translations_count", len(translations_df))

        # Step 1b: Validate loaded data
        with tracker.stage("validate_data"):
            if PANDERA_AVAILABLE and faq_schema is not None:
                faq_df = validate_dataframe(faq_df, faq_schema, "FAQ")
                if not translations_df.empty:
                    translations_df = validate_dataframe(translations_df, translation_schema, "Translations")
            tracker.metric("faq_valid", len(faq_df))
            tracker.metric("translations_valid", len(translations_df))

        # Step 2: EDA
        with tracker.stage("eda"):
            stats_df = visualize_data_composition(faq_df, pdf_chunks, teacher_qa_df, translations_df)
            visualize_text_distributions(faq_df, pdf_chunks, teacher_qa_df, translations_df)
            visualize_term_frequencies(faq_df, pdf_chunks, translations_df)
            create_interactive_visualizations(faq_df, pdf_chunks, translations_df)
            visualize_quality_metrics(faq_df, teacher_qa_df)
            generate_eda_report(faq_df, pdf_chunks, teacher_qa_df, translations_df)

        # Step 3: Generate teacher QA
        with tracker.stage("teacher_qa_generation"):
            cached_qa = checkpoint.load("teacher_qa_generated")
            if cached_qa is not None:
                teacher_qa_generated = cached_qa
                logger.info("Loaded teacher QA from checkpoint")
            else:
                teacher_qa_generated = []
                if pdf_chunks:
                    teacher_qa_generated = generate_enhanced_teacher_qa(pdf_chunks, config)
                checkpoint.save("teacher_qa_generated", teacher_qa_generated)

            all_teacher_qa = []
            if not teacher_qa_df.empty:
                all_teacher_qa.extend(teacher_qa_df.to_dict("records"))
            all_teacher_qa.extend(teacher_qa_generated)
            tracker.metric("total_teacher_qa", len(all_teacher_qa))

        # Step 4: Augment data
        with tracker.stage("augmentation"):
            augmented_data = augmentor.augment(faq_df, all_teacher_qa, translations_df)
            if not augmented_data:
                logger.error("No augmented data generated!")
                return
            tracker.metric("augmented_samples", len(augmented_data))

        # Step 5: Export datasets
        with tracker.stage("export"):
            output_file = config.output_dir / "enhanced_training_data.jsonl"
            exporter.export_jsonl(augmented_data, output_file)

            split_dir = config.output_dir / "enhanced_splits"
            exporter.export_train_val_split(augmented_data, split_dir)

            stats_file = config.output_dir / "enhanced_dataset_statistics.json"
            stats = exporter.export_statistics(augmented_data, stats_file)

            translation_data = [d for d in augmented_data if d.get("data_type") == DataType.TRANSLATION.value]
            if translation_data:
                exporter.export_jsonl(translation_data, config.output_dir / "translation_dataset.jsonl")

            teacher_qa_data = [d for d in augmented_data if d.get("data_type") == DataType.TEACHER_QA.value]
            if teacher_qa_data:
                exporter.export_jsonl(teacher_qa_data, config.output_dir / "teacher_qa_dataset.jsonl")

        # Pipeline summary
        summary = tracker.summary()
        summary_path = config.output_dir / "pipeline_summary.json"
        with open(summary_path, "w") as f:
            json.dump(summary, f, indent=2, default=str)

        logger.info("\n" + "=" * 70)
        logger.info("PIPELINE WITH EDA COMPLETE")
        logger.info("=" * 70)
        logger.info(f"Total time: {summary['total_elapsed_s']:.1f}s")
        logger.info(f"Total samples: {len(augmented_data)}")
        logger.info(f"Output: {output_file}")
        logger.info(f"EDA: {eda_dir}")

        # List EDA files
        logger.info("\nEDA files generated:")
        for file in sorted(eda_dir.glob("*")):
            if file.is_file():
                size_kb = file.stat().st_size / 1024
                logger.info(f"  {file.name} ({size_kb:.1f} KB)")

    except Exception as e:
        logger.error(f"Pipeline with EDA failed: {e}", exc_info=True)
        raise

# Run the enhanced pipeline with EDA
if __name__ == "__main__":
    enhanced_main_pipeline_with_eda()
